# El Mundo

In [ ]:
import requests
from bs4 import BeautifulSoup
import re


# settings

url = (
    "https://www.elmundo.es/elmundo/hemeroteca/2022/02/24/t/index.html"
)


# headers

headers = {

    "User-Agent": (

        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"

    )

}


# fetch

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

html = response.text


print("\nstatus\n")

print(
    response.status_code
)


print("\nhtml size\n")

print(
    len(html)
)


# soup

soup = BeautifulSoup(
    html,
    "html.parser"
)


# all links

links = soup.find_all(
    "a",
    href=True
)


all_urls = []


for link in links:

    href = link["href"]

    all_urls.append(
        href
    )


# deduplicate

all_urls = list(
    set(all_urls)
)


print("\ntotal unique links\n")

print(
    len(all_urls)
)


# article detection

article_urls = []


for url in all_urls:

    lower = url.lower()


    if re.search(

        r"/20\d{2}/\d{2}/\d{2}/",

        lower

    ):

        article_urls.append(
            url
        )


print("\narticle urls found\n")

print(
    len(article_urls)
)


# sample urls

print("\nsample article urls\n")

for u in article_urls[:100]:

    print(u)


# image check

images = soup.find_all("img")


print("\nimages found\n")

print(
    len(images)
)


# sample image urls

print("\nsample image urls\n")

for img in images[:20]:

    src = img.get("src")

    if src:

        print(src)


print("\nfinished")

In [ ]:
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import pandas as pd
import re
import time
from datetime import datetime
from datetime import timedelta


# settings

start_date = "2022-01-01"
end_date = "2022-01-31"

max_days = 10

sample_per_section = 5


# headers

headers = {

    "User-Agent": (

        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"

    )

}


session = requests.Session()

session.headers.update(headers)


# generate dates

start_dt = datetime.strptime(
    start_date,
    "%Y-%m-%d"
)

end_dt = datetime.strptime(
    end_date,
    "%Y-%m-%d"
)

all_dates = []

current = start_dt

while current <= end_dt:

    all_dates.append(current)

    current += timedelta(days=1)


# optional limit

all_dates = all_dates[:max_days]


print("\narchive days\n")

print(
    len(all_dates)
)


# build archive urls

archive_urls = []


for dt in all_dates:

    archive_url = (

        "https://www.elmundo.es/"
        "elmundo/hemeroteca/"
        f"{dt.year}/"
        f"{dt.month:02d}/"
        f"{dt.day:02d}/"
        "t/index.html"

    )


    archive_urls.append({

        "date": dt.strftime("%Y-%m-%d"),

        "year": dt.year,

        "archive_url": archive_url

    })


# fetch helper

def fetch_html(url):

    response = session.get(
        url,
        timeout=30
    )

    response.raise_for_status()

    return response.text


# collect articles

all_articles = []

seen_urls = set()


for row in tqdm(archive_urls):

    archive_url = row["archive_url"]

    archive_date = row["date"]

    archive_year = row["year"]


    try:

        html = fetch_html(
            archive_url
        )

        soup = BeautifulSoup(
            html,
            "html.parser"
        )


        links = soup.find_all(
            "a",
            href=True
        )


        for link in links:

            url = link["href"]


            if not isinstance(url, str):
                continue


            url = url.strip()


            # only elmundo

            if not url.startswith(
                "https://www.elmundo.es/"
            ):
                continue


            # remove anchors

            url = url.split("#")[0]


            # remove params

            url = url.split("?")[0]


            # article pattern

            if not re.search(

                r"/20\d{2}/\d{2}/\d{2}/",

                url

            ):
                continue


            # must end with html

            if not url.endswith(".html"):
                continue


            # remove duplicates

            if url in seen_urls:
                continue


            seen_urls.add(url)


            # path parts

            path_parts = [

                p
                for p in url
                .replace(
                    "https://www.elmundo.es/",
                    ""
                )
                .split("/")
                if p

            ]


            if len(path_parts) < 4:
                continue


            section = path_parts[0]


            all_articles.append({

                "archive_date": archive_date,

                "archive_year": archive_year,

                "section": section,

                "article_url": url

            })


        time.sleep(0.3)


    except Exception as e:

        print("\narchive error")

        print(archive_url)

        print(e)


# dataframe

articles_df = pd.DataFrame(
    all_articles
)


# diagnostics

print("\narticle urls\n")

print(
    len(articles_df)
)


print("\nunique sections\n")

print(
    articles_df["section"]
    .nunique()
)


print("\nsection distribution\n")

section_counts = (

    articles_df["section"]
    .value_counts()

)

print(
    section_counts
)


# sample urls per section

print("\nsample urls by section\n")


for section in section_counts.index:

    print("\n" + "=" * 80)

    print(section)

    print("=" * 80)


    sample_urls = (

        articles_df[
            articles_df["section"] == section
        ]["article_url"]

        .head(sample_per_section)

    )


    for url in sample_urls:

        print(url)


# save diagnostics

articles_df.to_csv(

    "elmundo_section_diagnostics.csv",

    index=False,

    encoding="utf-8-sig"

)


print("\ndiagnostics saved")

print(
    "elmundo_section_diagnostics.csv"
)

print("\nfinished")

In [ ]:
import re
import csv
import asyncio
import aiohttp

from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from datetime import datetime
from datetime import timedelta


# settings

headers = {
    "User-Agent": "Mozilla/5.0"
}

start_date = "2022-01-01"
end_date = "2026-12-31"

concurrency = 50

output_csv = "elmundo_corpus.csv"


# target sections

allowed_sections = {

    "internacional",
    "espana",
    "opinion",
    "economia",
    "sociedad",
    "cronica",
    "papel",
    "madrid",
    "cataluna",
    "comunidad-valenciana",
    "andalucia",
    "pais-vasco",
    "baleares"

}


# csv

csv_file = open(
    output_csv,
    "w",
    newline="",
    encoding="utf-8-sig"
)

writer = csv.DictWriter(
    csv_file,
    fieldnames=[
        "archive_date",
        "section",
        "url",
        "title",
        "published_time",
        "text",
        "image_count",
        "image_urls",
    ]
)

writer.writeheader()


# dates

start_dt = datetime.strptime(
    start_date,
    "%Y-%m-%d"
)

end_dt = datetime.strptime(
    end_date,
    "%Y-%m-%d"
)

dates = []

current = start_dt

while current <= end_dt:

    dates.append(current)

    current += timedelta(days=1)


print("\narchive days\n")

print(len(dates))


# helpers

def build_archive_url(dt):

    return (

        "https://www.elmundo.es/"
        "elmundo/hemeroteca/"
        f"{dt.year}/"
        f"{dt.month:02d}/"
        f"{dt.day:02d}/"
        "t/index.html"

    )


def extract_section(url):

    try:

        path = url.replace(
            "https://www.elmundo.es/",
            ""
        )

        return path.split("/")[0]

    except:

        return None


def clean_url(url):

    if not url:
        return None


    if not isinstance(url, str):
        return None


    lower = url.lower()


    # must look like image

    if not any(

        ext in lower

        for ext in [

            ".jpg",
            ".jpeg",
            ".png",
            ".webp",
            ".avif"

        ]

    ):

        return None


    # remove obvious junk

    bad_terms = [

        "logo",
        "icon",
        "avatar",
        "tracking",
        "pixel",
        "sprite",
        "favicon",
        "ads",
        "banner",
        "button"

    ]


    if any(

        bad in lower

        for bad in bad_terms

    ):

        return None


    # allow relative urls

    if lower.startswith("//"):

        url = "https:" + url


    elif lower.startswith("/"):

        url = "https://www.spiegel.de" + url


    return url


def is_article_url(url):

    pattern = r"/20\d{2}/\d{2}/\d{2}/"

    if not re.search(pattern, url):
        return False

    if not url.endswith(".html"):
        return False

    return True


def extract_image_urls(soup):

    images = set()


    for img in soup.find_all("img"):

        src = img.get("src")

        if not src:
            continue

        if not src.startswith("http"):
            continue


        low = src.lower()


        bad = [

            ".svg",
            "logo",
            "icon",
            "avatar",
            "pixel",
            "ads",
            "banner"

        ]


        if any(x in low for x in bad):
            continue


        images.add(src)


    og = soup.find(
        "meta",
        property="og:image"
    )

    if og:

        content = og.get("content")

        if content:

            images.add(content)


    return list(images)


# fetch

async def fetch_html(
    session,
    url
):

    try:

        async with session.get(
            url,
            timeout=20
        ) as r:

            if r.status != 200:
                return None

            return await r.text()

    except:

        return None


# archive parser

async def extract_article_urls(
    session,
    archive_url
):

    html = await fetch_html(
        session,
        archive_url
    )

    if not html:
        return []


    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    found = set()


    for a in soup.find_all(
        "a",
        href=True
    ):

        url = a["href"]


        if not isinstance(url, str):
            continue


        if not url.startswith(
            "https://www.elmundo.es/"
        ):
            continue


        url = clean_url(url)


        if not is_article_url(url):
            continue


        section = extract_section(url)

        if section not in allowed_sections:
            continue


        found.add(url)


    return list(found)


# metadata extraction

async def extract_article_data(
    session,
    url,
    archive_date
):

    try:

        html = await fetch_html(
            session,
            url
        )

        if not html:
            return None


        soup = BeautifulSoup(
            html,
            "html.parser"
        )


        # title

        title = None

        h1 = soup.find("h1")

        if h1:

            title = h1.get_text(
                strip=True
            )


        # publication time

        published_time = None

        meta_time = soup.find(

            "meta",

            property="article:published_time"

        )

        if meta_time:

            published_time = meta_time.get(
                "content"
            )


        # text

        paragraphs = []


        for p in soup.find_all("p"):

            text = p.get_text(
                " ",
                strip=True
            )


            if len(text) < 40:
                continue


            paragraphs.append(text)


        article_text = "\n".join(
            paragraphs
        )


        # images

        image_urls = extract_image_urls(
            soup
        )


        return {

            "archive_date": archive_date,

            "section": extract_section(url),

            "url": url,

            "title": title,

            "published_time": published_time,

            "text": article_text,

            "image_count": len(image_urls),

            "image_urls": " | ".join(
                image_urls
            )

        }

    except Exception as e:

        print("\narticle error")

        print(url)

        print(e)

        return None


# worker

async def process_article(

    session,
    semaphore,
    url,
    archive_date,
    pbar

):

    async with semaphore:

        data = await extract_article_data(

            session,
            url,
            archive_date

        )

        if data:

            writer.writerow(data)

            pbar.update(1)


# main

async def main():

    connector = aiohttp.TCPConnector(
        limit=concurrency
    )

    timeout = aiohttp.ClientTimeout(
        total=30
    )

    semaphore = asyncio.Semaphore(
        concurrency
    )

    pbar = tqdm(
        desc="articles",
        unit="article"
    )

    async with aiohttp.ClientSession(

        headers=headers,
        connector=connector,
        timeout=timeout

    ) as session:


        # step 1
        # collect article urls

        print("\ncollecting urls\n")


        all_urls = []


        for dt in tqdm(dates):

            archive_url = build_archive_url(dt)

            urls = await extract_article_urls(

                session,
                archive_url

            )


            for url in urls:

                all_urls.append({

                    "url": url,

                    "archive_date":
                    dt.strftime("%Y-%m-%d")

                })


        print("\nurls found\n")

        print(len(all_urls))


        # deduplicate

        seen = set()

        unique_urls = []


        for item in all_urls:

            if item["url"] in seen:
                continue

            seen.add(item["url"])

            unique_urls.append(item)


        print("\nunique urls\n")

        print(len(unique_urls))


        # step 2
        # metadata extraction

        print("\nextracting metadata\n")


        tasks = []


        for item in unique_urls:

            task = asyncio.create_task(

                process_article(

                    session,
                    semaphore,
                    item["url"],
                    item["archive_date"],
                    pbar

                )

            )

            tasks.append(task)


        await asyncio.gather(*tasks)


    pbar.close()

    csv_file.close()


    print("\ncorpus saved")

    print(output_csv)

    print("\nfinished")


# jupyter

await main()

In [ ]:
import pandas as pd
import re


# load corpus

df = pd.read_csv(
    "elmundo_corpus.csv"
)


print("\ninput corpus\n")

print(len(df))


# clean text fields

df["title"] = (
    df["title"]
    .fillna("")
    .astype(str)
)

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
)


# combined text

df["combined"] = (

    df["title"]
    + " "
    + df["text"]

).str.lower()


# high recall ukraine-russia lexicon

patterns = [

    # countries and leaders

    r"\bucrani\w*",
    r"\brusi\w*",
    r"\bmoscu\w*",
    r"\bmosk\w*",
    r"\bkremlin\w*",
    r"\bputin\w*",
    r"\bzelens\w*",
    r"\bzelensk\w*",
    r"\bzelenski\w*",
    r"\bzelensky\w*",
    r"\bmedvedev\w*",
    r"\blavrov\w*",
    r"\bshoigu\w*",
    r"\bprigozh\w*",
    r"\bwagner\w*",
    r"\blukashenko\w*",
    r"\bbielorrus\w*",


    # territories and cities

    r"\bdonbas\w*",
    r"\bdonetsk\w*",
    r"\blugansk\w*",
    r"\bcrime\w*",
    r"\bsebastopol\w*",
    r"\bmariupol\w*",
    r"\bodesa\w*",
    r"\bkherson\w*",
    r"\bkhark\w*",
    r"\bkiev\w*",
    r"\bkyiv\w*",
    r"\bzapori\w*",
    r"\bchernobil\w*",
    r"\bchernihiv\w*",
    r"\bbakhmut\w*",


    # war terminology

    r"\binvasion\w*",
    r"\bofensiv\w*",
    r"\bcontraofensiv\w*",
    r"\bbombarde\w*",
    r"\bmisil\w*",
    r"\bdron\w*",
    r"\bataqu\w*",
    r"\bejercit\w*",
    r"\btrop\w*",
    r"\bmercenario\w*",
    r"\bmovilizacion\w*",
    r"\banexion\w*",
    r"\bfrent\w*",
    r"\btrincher\w*",
    r"\bocupacion\w*",
    r"\bdespliegue militar\w*",
    r"\barmament\w*",
    r"\bdefensa aerea\w*",


    # geopolitical

    r"\botan\w*",
    r"\bnato\w*",
    r"\bsancion\w*",
    r"\bunion europea\w*",
    r"\bue\w*",
    r"\barmas occidentales\w*",
    r"\bayuda militar\w*",
    r"\bpaquete militar\w*",
    r"\bcorredor humanitario\w*",
    r"\bcrisis energetica\w*",
    r"\bgas ruso\w*",
    r"\bpetroleo ruso\w*",
    r"\benergia rusa\w*",


    # nuclear

    r"\bnuclear\w*",
    r"\bamenaza nuclear\w*",
    r"\bcentral nuclear\w*",
    r"\bzaporizhzhia\w*",


    # refugees and civilians

    r"\brefugiad\w*",
    r"\bdesplazad\w*",
    r"\bevacuacion\w*",
    r"\bcivil\w*",


    # propaganda and cyber

    r"\bdesinformacion\w*",
    r"\bpropaganda rusa\w*",
    r"\bfake news\w*",
    r"\bciberguerra\w*",
    r"\bataque cibernetico\w*",

]


combined_pattern = "|".join(patterns)


# filter corpus

mask = df["combined"].str.contains(

    combined_pattern,
    regex=True,
    na=False

)

filtered = df[mask].copy()


print("\nfiltered corpus\n")

print(len(filtered))


# deduplicate

if "url" in filtered.columns:

    filtered = filtered.drop_duplicates(
        subset=["url"]
    )

elif "article_url" in filtered.columns:

    filtered = filtered.drop_duplicates(
        subset=["article_url"]
    )


print("\nafter dedup\n")

print(len(filtered))


# section distribution

print("\nsection distribution\n")


if "section" in filtered.columns:

    print(

        filtered["section"]
        .value_counts()

    )

elif "section_article" in filtered.columns:

    print(

        filtered["section_article"]
        .value_counts()

    )


# year distribution

print("\nyear distribution\n")


years = pd.to_datetime(

    filtered["published_time"],

    errors="coerce",

    utc=True

)


print(

    years.dt.year
    .value_counts()
    .sort_index()

)


# image statistics

if "image_count" in filtered.columns:

    print("\nimage count distribution\n")

    print(

        filtered["image_count"]
        .describe()

    )


# articles with images

if "image_count" in filtered.columns:

    filtered["has_images"] = (
        filtered["image_count"] > 0
    )

    print("\narticles with images\n")

    print(

        filtered["has_images"]
        .value_counts()

    )


# live pages

url_column = None

if "url" in filtered.columns:
    url_column = "url"

elif "article_url" in filtered.columns:
    url_column = "article_url"


if url_column is not None:

    filtered["is_live"] = (

        filtered[url_column]
        .str.contains(
            "directo",
            case=False,
            na=False
        )

    )

    print("\nlive pages\n")

    print(

        filtered["is_live"]
        .value_counts()

    )


# save corpus

filtered.to_csv(

    "elmundo_ukraine_russia_high_recall.csv",

    index=False,

    encoding="utf-8-sig"

)


print("\nfiltered corpus saved\n")

print("elmundo_ukraine_russia_high_recall.csv")


print("\nfinished")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import Counter


# settings

DATASET = "elmundo_ukraine_russia_high_recall.csv"


# load

df = pd.read_csv(DATASET)


print("\ninput rows\n")

print(len(df))


# clean

df["title"] = (
    df["title"]
    .fillna("")
    .astype(str)
)

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
)


# datetime

df["published_time"] = pd.to_datetime(

    df["published_time"],

    errors="coerce",

    utc=True

)


df["year"] = (
    df["published_time"]
    .dt.year
)

df["month"] = (
    df["published_time"]
    .dt.to_period("M")
    .astype(str)
)


# article length

df["text_length"] = (
    df["text"]
    .str.len()
)

df["word_count"] = (
    df["text"]
    .str.split()
    .str.len()
)


# live pages

url_column = None

if "url" in df.columns:
    url_column = "url"

elif "article_url" in df.columns:
    url_column = "article_url"


df["is_live"] = (

    df[url_column]
    .str.contains(
        "directo",
        case=False,
        na=False
    )

)


# section column

section_column = None

if "section" in df.columns:
    section_column = "section"

elif "section_article" in df.columns:
    section_column = "section_article"


# diagnostics

print("\nsection distribution\n")

print(

    df[section_column]
    .value_counts()
    .head(20)

)


print("\nyear distribution\n")

print(

    df["year"]
    .value_counts()
    .sort_index()

)


print("\nlive pages\n")

print(

    df["is_live"]
    .value_counts()

)


print("\narticle length stats\n")

print(

    df["word_count"]
    .describe()

)


print("\nimage count stats\n")

print(

    df["image_count"]
    .describe()

)


# monthly publication peaks

monthly_counts = (

    df.groupby("month")
    .size()
    .sort_index()

)


print("\nmonthly peaks\n")

print(

    monthly_counts
    .sort_values(ascending=False)
    .head(20)

)


# top title words

stopwords = {

    "de",
    "la",
    "el",
    "los",
    "las",
    "y",
    "en",
    "del",
    "que",
    "un",
    "una",
    "con",
    "por",
    "para",
    "a",
    "al",
    "se",
    "su",
    "sus",
    "como",
    "tras",
    "sobre",
    "desde",
    "ante",
    "entre",
    "más",
    "menos"

}


all_words = []


for title in df["title"]:

    words = re.findall(

        r"\b\w+\b",

        title.lower()

    )

    words = [

        w for w in words

        if len(w) > 3
        and w not in stopwords

    ]

    all_words.extend(words)


word_counts = Counter(all_words)


print("\ntop title words\n")

for word, count in word_counts.most_common(50):

    print(word, count)


# image domains

if "image_urls" in df.columns:

    image_domains = []


    for row in df["image_urls"].dropna():

        try:

            urls = eval(row)

            for u in urls:

                m = re.search(

                    r"https?://([^/]+)",

                    str(u)

                )

                if m:
                    image_domains.append(
                        m.group(1)
                    )

        except:
            pass


    domain_counts = Counter(image_domains)


    print("\nimage domains\n")

    for domain, count in domain_counts.most_common(20):

        print(domain, count)


# plots

plt.figure(figsize=(14, 6))

monthly_counts.plot()

plt.title("monthly publication volume")

plt.xticks(rotation=90)

plt.tight_layout()

plt.show()


plt.figure(figsize=(10, 6))

df[section_column].value_counts().head(15).plot(
    kind="bar"
)

plt.title("top sections")

plt.tight_layout()

plt.show()


plt.figure(figsize=(10, 6))

df["year"].value_counts().sort_index().plot(
    kind="bar"
)

plt.title("year distribution")

plt.tight_layout()

plt.show()


# save diagnostics

monthly_counts.to_csv(
    "elmundo_monthly_counts.csv"
)


print("\ndiagnostics saved\n")

print("elmundo_monthly_counts.csv")


print("\nfinished")

## Image download 

In [ ]:
import pandas as pd
import requests
import os
import hashlib
import time
import random
import ast

from tqdm import tqdm
from urllib.parse import urlparse


# settings

INPUT_DATASET = "elmundo_ukraine_russia_high_recall.csv"

OUTPUT_DIR = "elmundo_images"

MAX_IMAGES_PER_ARTICLE = None

ONLY_FIRST_IMAGE = False

START_INDEX = 0

SLEEP_RANGE = (0.1, 0.4)

TIMEOUT = 30


# create folders

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# load dataset

df = pd.read_csv(
    INPUT_DATASET
)


print("\ninput rows\n")

print(len(df))


# parse image urls

def parse_images(x):

    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    try:
        return ast.literal_eval(x)

    except:
        return []


df["image_urls"] = (
    df["image_urls"]
    .apply(parse_images)
)


# records

downloaded = []

failed = []


# headers

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )

}


# download loop

for idx, row in tqdm(

    df.iterrows(),

    total=len(df)

):

    if idx < START_INDEX:
        continue


    article_url = row.get(
        "url",
        row.get("article_url", "")
    )

    title = row.get("title", "")

    section = row.get(
        "section",
        row.get("section_article", "")
    )

    published_time = row.get(
        "published_time",
        ""
    )

    image_urls = row["image_urls"]


    if ONLY_FIRST_IMAGE:
        image_urls = image_urls[:1]


    if MAX_IMAGES_PER_ARTICLE is not None:

        image_urls = image_urls[
            :MAX_IMAGES_PER_ARTICLE
        ]


    for image_idx, image_url in enumerate(image_urls):

        try:

            parsed = urlparse(image_url)

            ext = os.path.splitext(
                parsed.path
            )[1].lower()


            if ext not in [

                ".jpg",
                ".jpeg",
                ".png",
                ".webp",
                ".gif"

            ]:

                ext = ".jpg"


            image_hash = hashlib.md5(

                image_url.encode()

            ).hexdigest()


            filename = (
                f"{image_hash}{ext}"
            )


            section_folder = os.path.join(
                OUTPUT_DIR,
                str(section)
            )


            os.makedirs(
                section_folder,
                exist_ok=True
            )


            filepath = os.path.join(
                section_folder,
                filename
            )


            if os.path.exists(filepath):

                downloaded.append({

                    "article_url": article_url,
                    "title": title,
                    "section": section,
                    "published_time": published_time,
                    "image_url": image_url,
                    "filepath": filepath,
                    "status": "already_exists"

                })

                continue


            response = requests.get(

                image_url,

                headers=HEADERS,

                timeout=TIMEOUT,

                stream=True

            )


            response.raise_for_status()


            with open(filepath, "wb") as f:

                for chunk in response.iter_content(
                    8192
                ):

                    f.write(chunk)


            downloaded.append({

                "article_url": article_url,
                "title": title,
                "section": section,
                "published_time": published_time,
                "image_url": image_url,
                "filepath": filepath,
                "status": "downloaded"

            })


            time.sleep(

                random.uniform(
                    *SLEEP_RANGE
                )

            )


        except Exception as e:

            failed.append({

                "article_url": article_url,
                "image_url": image_url,
                "error": str(e)

            })

            print("\ndownload error\n")

            print(image_url)

            print(e)


# save logs

downloaded_df = pd.DataFrame(
    downloaded
)

failed_df = pd.DataFrame(
    failed
)


downloaded_df.to_csv(

    "elmundo_downloaded_images.csv",

    index=False,

    encoding="utf-8-sig"

)


failed_df.to_csv(

    "elmundo_failed_images.csv",

    index=False,

    encoding="utf-8-sig"

)


# final stats

print("\ndownload finished\n")

print(

    "images downloaded:",

    len(downloaded_df)

)

print(

    "failed images:",

    len(failed_df)

)


print("\nfiles saved\n")

print("elmundo_downloaded_images.csv")

print("elmundo_failed_images.csv")


print("\nfinished")

# Die Welt

In [ ]:
import requests
import gzip

from io import BytesIO
from lxml import etree


SITEMAP_URL = (
    "https://www.welt.de/sitemaps/sitemap/2022/01/sitemap.xml.gz"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}


def load_sitemap(url):

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=60
    )

    response.raise_for_status()

    xml_bytes = gzip.decompress(
        response.content
    )

    tree = etree.parse(
        BytesIO(xml_bytes)
    )

    return tree


tree = load_sitemap(
    SITEMAP_URL
)

root = tree.getroot()

print("\nROOT TAG:")
print(root.tag)

print("\nNAMESPACES:")

for k, v in root.nsmap.items():
    print(f"{k}: {v}")

urls = root.findall(
    "{http://www.sitemaps.org/schemas/sitemap/0.9}url"
)

print(f"\nTOTAL URLS: {len(urls)}")

In [ ]:
from lxml import etree

print(etree.tostring(
    urls[0],
    pretty_print=True,
    encoding="unicode"
))

In [ ]:
import requests
from bs4 import BeautifulSoup
import json

TEST_URL = "https://www.welt.de/politik/deutschland/article6a1fbb93639e3d502fd49706/ukraine-beschiesst-st-petersburg-kurz-vor-putin-forum-cdu-mann-bezeichnet-afd-gaeste-als-dumm.html"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}


response = requests.get(
    TEST_URL,
    headers=HEADERS,
    timeout=60
)

response.raise_for_status()

soup = BeautifulSoup(
    response.text,
    "lxml"
)

jsonld_blocks = soup.find_all(
    "script",
    type="application/ld+json"
)

print(f"\nFound JSON-LD blocks: {len(jsonld_blocks)}")

for block_num, script in enumerate(jsonld_blocks):

    try:

        data = json.loads(
            script.get_text(strip=True)
        )

        if (
            isinstance(data, dict)
            and data.get("@type") == "NewsArticle"
        ):

            print("\n" + "=" * 80)
            print("NEWS ARTICLE FOUND")
            print("=" * 80)

            print("\nHEADLINE:")
            print(data.get("headline"))

            print("\nDATE PUBLISHED:")
            print(data.get("datePublished"))

            print("\nAUTHOR:")
            print(data.get("author"))

            print("\nIMAGE TYPE:")
            print(type(data.get("image")))

            print("\nIMAGE:")
            print(data.get("image"))

            print("\nARTICLE BODY LENGTH:")

            body = data.get(
                "articleBody",
                ""
            )

            print(len(body))

            print("\nFIRST 500 CHARS:")
            print(body[:500])

            break

    except Exception as e:

        print(
            f"JSON ERROR: {e}"
        )

### Test 1 month

In [ ]:
import requests
import gzip
import json
import pandas as pd

from io import BytesIO
from lxml import etree
from bs4 import BeautifulSoup
from tqdm import tqdm


SITEMAP_URL = (
    "https://www.welt.de/sitemaps/sitemap/2022/01/sitemap.xml.gz"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

ALLOWED_CATEGORIES = {
    "politik",
    "wirtschaft",
    "debatte",
    "vermischtes",
    "wissenschaft",
    "geschichte"
}


def load_urls():

    response = requests.get(
        SITEMAP_URL,
        headers=HEADERS,
        timeout=60
    )

    response.raise_for_status()

    xml_data = gzip.decompress(
        response.content
    )

    tree = etree.parse(
        BytesIO(xml_data)
    )

    ns = {
        "sm":
        "http://www.sitemaps.org/schemas/sitemap/0.9"
    }

    urls = tree.xpath(
        "//sm:url/sm:loc/text()",
        namespaces=ns
    )

    return urls


def filter_urls(urls):

    result = []

    for url in urls:

        try:

            path = url.replace(
                "https://www.welt.de/",
                ""
            )

            category = path.split("/")[0]

            if category in ALLOWED_CATEGORIES:

                result.append(
                    (url, category)
                )

        except Exception:
            pass

    return result


def extract_text(url):

    try:

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=60
        )

        if response.status_code != 200:
            return None

        soup = BeautifulSoup(
            response.text,
            "lxml"
        )

        scripts = soup.find_all(
            "script",
            type="application/ld+json"
        )

        for script in scripts:

            try:

                data = json.loads(
                    script.get_text(strip=True)
                )

                if (
                    isinstance(data, dict)
                    and data.get("@type")
                    == "NewsArticle"
                ):

                    return data.get(
                        "articleBody"
                    )

            except Exception:
                continue

        return None

    except Exception:
        return None


urls = load_urls()

print(
    f"Total sitemap URLs: {len(urls)}"
)

filtered = filter_urls(urls)

print(
    f"Filtered URLs: {len(filtered)}"
)

rows = []

for url, category in tqdm(filtered):

    text = extract_text(url)

    if not text:
        continue

    rows.append({
        "url": url,
        "category": category,
        "text": text
    })

df = pd.DataFrame(rows)

df.to_csv(
    "welt_jan2022_text_only.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nDONE")

print(
    f"Articles: {len(df)}"
)

print(
    f"Mean text length: "
    f"{df['text'].str.len().mean():.0f}"
)

print(
    "\nCategory distribution:"
)

print(
    df["category"]
    .value_counts()
)

### Downlad metadata

In [ ]:
import os
import json
import gzip
import requests
import pandas as pd

from io import BytesIO
from lxml import etree
from bs4 import BeautifulSoup
from tqdm import tqdm

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

# =====================================================
# SETTINGS
# =====================================================

START_YEAR = 2022
START_MONTH = 2

END_YEAR = 2025
END_MONTH = 12

MAX_WORKERS = 10

OUTPUT_CORPUS = "welt_full_text_corpus.csv"
OUTPUT_FAILED = "welt_all_failed.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

ALLOWED_CATEGORIES = {
    "politik",
    "wirtschaft",
    "debatte",
    "vermischtes",
    "wissenschaft",
    "geschichte"
}

# =====================================================
# SITEMAP URL
# =====================================================

def make_sitemap_url(year, month):

    return (
        f"https://www.welt.de/sitemaps/sitemap/"
        f"{year}/{month:02d}/sitemap.xml.gz"
    )

# =====================================================
# LOAD URLS FROM SITEMAP
# =====================================================

def load_urls(sitemap_url):

    try:

        response = requests.get(
            sitemap_url,
            headers=HEADERS,
            timeout=60
        )

        if response.status_code != 200:
            return []

        xml_data = gzip.decompress(
            response.content
        )

        tree = etree.parse(
            BytesIO(xml_data)
        )

        ns = {
            "sm":
            "http://www.sitemaps.org/schemas/sitemap/0.9"
        }

        urls = tree.xpath(
            "//sm:url/sm:loc/text()",
            namespaces=ns
        )

        return urls

    except Exception as e:

        print(
            f"ERROR SITEMAP: {sitemap_url}"
        )

        print(e)

        return []

# =====================================================
# CATEGORY FILTER
# =====================================================

def filter_urls(urls):

    filtered = []

    for url in urls:

        try:

            path = url.replace(
                "https://www.welt.de/",
                ""
            )

            category = path.split("/")[0]

            if category in ALLOWED_CATEGORIES:

                filtered.append(
                    (url, category)
                )

        except Exception:
            pass

    return filtered

# =====================================================
# ARTICLE EXTRACTOR
# =====================================================

def extract_article(url):

    try:

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=60
        )

        if response.status_code != 200:

            return (
                False,
                None,
                f"HTTP_{response.status_code}"
            )

        soup = BeautifulSoup(
            response.text,
            "lxml"
        )

        scripts = soup.find_all(
            "script",
            type="application/ld+json"
        )

        for script in scripts:

            try:

                data = json.loads(
                    script.get_text(strip=True)
                )

                if (
                    isinstance(data, dict)
                    and data.get("@type")
                    == "NewsArticle"
                ):

                    text = data.get(
                        "articleBody"
                    )

                    if not text:

                        return (
                            False,
                            None,
                            "EMPTY_ARTICLEBODY"
                        )

                    return (
                        True,
                        text,
                        None
                    )

            except Exception:
                continue

        return (
            False,
            None,
            "NO_NEWSARTICLE_JSONLD"
        )

    except Exception as e:

        return (
            False,
            None,
            type(e).__name__
        )

# =====================================================
# BUILD URL LIST
# =====================================================

all_urls = []

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    start_month = 1
    end_month = 12

    if year == START_YEAR:
        start_month = START_MONTH

    if year == END_YEAR:
        end_month = END_MONTH

    for month in range(
        start_month,
        end_month + 1
    ):

        sitemap_url = make_sitemap_url(
            year,
            month
        )

        print(
            f"\nLoading {year}-{month:02d}"
        )

        urls = load_urls(
            sitemap_url
        )

        filtered = filter_urls(
            urls
        )

        print(
            f"Found {len(filtered)} URLs"
        )

        all_urls.extend(
            filtered
        )

# =====================================================
# REMOVE DUPLICATES
# =====================================================

seen = set()
unique_urls = []

for url, category in all_urls:

    if url not in seen:

        seen.add(url)

        unique_urls.append(
            (url, category)
        )

print(
    f"\nTotal unique URLs: "
    f"{len(unique_urls):,}"
)

# =====================================================
# DOWNLOAD TEXTS
# =====================================================

success_rows = []
failed_rows = []

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = {

        executor.submit(
            extract_article,
            url
        ): (url, category)

        for url, category
        in unique_urls
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        url, category = futures[
            future
        ]

        try:

            success, text, reason = (
                future.result()
            )

            if success:

                success_rows.append({

                    "url":
                        url,

                    "category":
                        category,

                    "text":
                        text

                })

            else:

                failed_rows.append({

                    "url":
                        url,

                    "category":
                        category,

                    "reason":
                        reason

                })

        except Exception as e:

            failed_rows.append({

                "url":
                    url,

                "category":
                    category,

                "reason":
                    str(e)

            })

# =====================================================
# SAVE CORPUS
# =====================================================

corpus_df = pd.DataFrame(
    success_rows
)

failed_df = pd.DataFrame(
    failed_rows
)

corpus_df.to_csv(
    OUTPUT_CORPUS,
    index=False,
    encoding="utf-8-sig"
)

failed_df.to_csv(
    OUTPUT_FAILED,
    index=False,
    encoding="utf-8-sig"
)

# =====================================================
# DIAGNOSTICS
# =====================================================

print("\n" + "=" * 60)

print(
    f"Articles collected: "
    f"{len(corpus_df):,}"
)

print(
    f"Failed URLs: "
    f"{len(failed_df):,}"
)

if len(corpus_df) > 0:

    lengths = corpus_df[
        "text"
    ].str.len()

    print(
        f"Mean text length: "
        f"{lengths.mean():.0f}"
    )

    print(
        f"Texts > 500 chars: "
        f"{(lengths > 500).sum():,}"
    )

    print(
        f"Texts > 2000 chars: "
        f"{(lengths > 2000).sum():,}"
    )

    print(
        "\nCategory distribution:\n"
    )

    print(
        corpus_df[
            "category"
        ].value_counts()
    )

if len(failed_df) > 0:

    print(
        "\nFailure reasons:\n"
    )

    print(
        failed_df[
            "reason"
        ].value_counts()
    )

print("\nDONE")
print(f"\nSaved: {OUTPUT_CORPUS}")
print(f"Saved: {OUTPUT_FAILED}")

In [ ]:
import json
import requests
import pandas as pd

from bs4 import BeautifulSoup
from tqdm import tqdm

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

# =====================================================
# SETTINGS
# =====================================================

MAIN_CORPUS_FILE = "welt_full_text_corpus.csv"

FAILED_FILE = "welt_all_failed.csv"

NEW_FAILED_FILE = (
    "welt_all_failed_after_recovery.csv"
)

MAX_WORKERS = 10

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

# =====================================================
# ACCEPTED ARTICLE TYPES
# =====================================================

VALID_TYPES = {

    "NewsArticle",

    "OpinionNewsArticle",

    "AnalysisNewsArticle",

    "ReportageNewsArticle",

    "Article"

}

# =====================================================
# EXTRACT ARTICLE
# =====================================================

def extract_article(url):

    try:

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=60
        )

        if response.status_code != 200:

            return (
                False,
                None,
                f"HTTP_{response.status_code}"
            )

        soup = BeautifulSoup(
            response.text,
            "lxml"
        )

        scripts = soup.find_all(
            "script",
            type="application/ld+json"
        )

        for script in scripts:

            try:

                data = json.loads(
                    script.get_text(strip=True)
                )

            except Exception:
                continue

            if not isinstance(
                data,
                dict
            ):
                continue

            article_type = data.get(
                "@type"
            )

            if article_type not in VALID_TYPES:
                continue

            text = data.get(
                "articleBody"
            )

            if not text:

                return (
                    False,
                    None,
                    "EMPTY_ARTICLEBODY"
                )

            return (
                True,
                text,
                article_type
            )

        return (
            False,
            None,
            "NO_VALID_ARTICLE_TYPE"
        )

    except Exception as e:

        return (
            False,
            None,
            type(e).__name__
        )

# =====================================================
# LOAD FILES
# =====================================================

main_df = pd.read_csv(
    MAIN_CORPUS_FILE,
    low_memory=False
)

failed_df = pd.read_csv(
    FAILED_FILE,
    low_memory=False
)

print("\nMAIN CORPUS")

print(len(main_df))

print("\nFAILED URLS")

print(len(failed_df))

# =====================================================
# RECOVER ARTICLES
# =====================================================

recovered_rows = []

still_failed_rows = []

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = {

        executor.submit(
            extract_article,
            row["url"]
        ): row

        for _, row
        in failed_df.iterrows()
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        row = futures[future]

        success, text, info = (
            future.result()
        )

        if success:

            recovered_rows.append({

                "url":
                    row["url"],

                "category":
                    row["category"],

                "text":
                    text

            })

        else:

            still_failed_rows.append({

                "url":
                    row["url"],

                "category":
                    row["category"],

                "reason":
                    info

            })

# =====================================================
# MERGE WITH MAIN CORPUS
# =====================================================

recovered_df = pd.DataFrame(
    recovered_rows
)

combined_df = pd.concat(
    [
        main_df,
        recovered_df
    ],
    ignore_index=True
)

combined_df = combined_df.drop_duplicates(
    subset=["url"]
)

# =====================================================
# SAVE UPDATED MAIN CORPUS
# =====================================================

combined_df.to_csv(

    MAIN_CORPUS_FILE,

    index=False,

    encoding="utf-8-sig"
)

# =====================================================
# SAVE REMAINING FAILURES
# =====================================================

still_failed_df = pd.DataFrame(
    still_failed_rows
)

still_failed_df.to_csv(

    NEW_FAILED_FILE,

    index=False,

    encoding="utf-8-sig"
)

# =====================================================
# DIAGNOSTICS
# =====================================================

print("\n")

print(
    "=" * 60
)

print(
    "RECOVERY RESULTS"
)

print(
    "=" * 60
)

print(
    f"\nOriginal corpus: "
    f"{len(main_df):,}"
)

print(
    f"Recovered articles: "
    f"{len(recovered_df):,}"
)

print(
    f"Final corpus: "
    f"{len(combined_df):,}"
)

print(
    f"Still failed: "
    f"{len(still_failed_df):,}"
)

if len(recovered_df) > 0:

    print(
        "\nRecovered category distribution:\n"
    )

    print(
        recovered_df[
            "category"
        ].value_counts()
    )

if len(still_failed_df) > 0:

    print(
        "\nRemaining failure reasons:\n"
    )

    print(
        still_failed_df[
            "reason"
        ].value_counts()
    )

print("\nDONE")

print(
    f"\nUpdated corpus saved to:"
)

print(
    MAIN_CORPUS_FILE
)

print(
    f"\nRemaining failures saved to:"
)

print(
    NEW_FAILED_FILE
)

In [ ]:
import pandas as pd
import re

# =====================================================
# SETTINGS
# =====================================================

input_file = "welt_full_text_corpus.csv"

output_file = (
    "welt_ukraine_russia_high_recall.csv"
)

# =====================================================
# LOAD CORPUS
# =====================================================

df = pd.read_csv(
    input_file,
    low_memory=False
)

print("\ninput corpus\n")

print(len(df))

# =====================================================
# ANCHOR TERMS
# =====================================================

anchor_patterns = [

    r"ukrain\w*",
    r"ukrainisch\w*",
    r"ukrainer\w*",

    r"russland\w*",
    r"russisch\w*",
    r"russ\w*",
    r"russen\w*",

    r"putin\w*",

    r"selensky\w*",
    r"zelensky\w*",
    r"selens\w*",
    r"zelens\w*"

]

# =====================================================
# CONTEXT TERMS
# =====================================================

context_patterns = [

    r"krieg\w*",
    r"angriff\w*",
    r"invasion\w*",
    r"offensive\w*",

    r"nato\w*",
    r"waffen\w*",
    r"raket\w*",
    r"drohn\w*",
    r"panzer\w*",

    r"donbass\w*",
    r"donezk\w*",
    r"luhansk\w*",
    r"krim\w*",
    r"crimea\w*",

    r"kiew\w*",
    r"kyiv\w*",
    r"moskau\w*",

    r"kreml\w*",

    r"gaslieferung\w*",
    r"nord\s*stream\w*",
    r"energiekrise\w*"

]

# =====================================================
# COMPILE REGEX
# =====================================================

anchor_regex = re.compile(
    "|".join(anchor_patterns),
    flags=re.IGNORECASE
)

context_regex = re.compile(
    "|".join(context_patterns),
    flags=re.IGNORECASE
)

# =====================================================
# FILTER
# =====================================================

filtered = df[

    df["text"]
    .fillna("")
    .str.contains(
        anchor_regex,
        na=False
    )

    &

    df["text"]
    .fillna("")
    .str.contains(
        context_regex,
        na=False
    )

]

# =====================================================
# DEDUP
# =====================================================

filtered = filtered.drop_duplicates(
    subset=["url"]
)

# =====================================================
# DIAGNOSTICS
# =====================================================

print("\nfiltered corpus\n")

print(len(filtered))

print("\ncategory distribution\n")

print(

    filtered["category"]
    .value_counts()
    .head(30)

)

print("\ntext length distribution\n")

print(

    filtered["text"]
    .str.len()
    .describe()

)

# =====================================================
# SAVE
# =====================================================

filtered.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)

print("\nfiltered corpus saved\n")

print(output_file)

print("\nfinished")

### Metadata full

In [ ]:
import json
import requests
import pandas as pd

from bs4 import BeautifulSoup
from tqdm import tqdm

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

# =====================================================
# SETTINGS
# =====================================================

INPUT_FILE = (
    "welt_ukraine_russia_high_recall.csv"
)

OUTPUT_FILE = (
    "welt_ukraine_russia_with_metadata.csv"
)

FAILED_FILE = (
    "welt_metadata_failed.csv"
)

MAX_WORKERS = 10

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

# =====================================================
# AUTHOR PARSER
# =====================================================

def parse_author(author):

    if author is None:
        return ""

    if isinstance(author, str):
        return author

    if isinstance(author, dict):

        return author.get(
            "name",
            ""
        )

    if isinstance(author, list):

        names = []

        for item in author:

            if isinstance(item, dict):

                name = item.get(
                    "name"
                )

                if name:
                    names.append(
                        name
                    )

        return " | ".join(
            names
        )

    return str(author)

# =====================================================
# IMAGE PARSER
# =====================================================

def parse_images(image_field):

    image_urls = []

    image_captions = []

    if image_field is None:

        return (
            image_urls,
            image_captions
        )

    if isinstance(image_field, str):

        image_urls.append(
            image_field
        )

    elif isinstance(image_field, dict):

        url = image_field.get(
            "url"
        )

        if url:

            image_urls.append(
                url
            )

        caption = image_field.get(
            "caption"
        )

        if caption:

            image_captions.append(
                caption
            )

    elif isinstance(image_field, list):

        for item in image_field:

            if isinstance(item, str):

                image_urls.append(
                    item
                )

            elif isinstance(
                item,
                dict
            ):

                url = item.get(
                    "url"
                )

                if url:

                    image_urls.append(
                        url
                    )

                caption = item.get(
                    "caption"
                )

                if caption:

                    image_captions.append(
                        caption
                    )

    return (
        image_urls,
        image_captions
    )

# =====================================================
# EXTRACT METADATA
# =====================================================

def extract_metadata(url):

    try:

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=60
        )

        if response.status_code != 200:

            return (
                False,
                None,
                f"HTTP_{response.status_code}"
            )

        soup = BeautifulSoup(
            response.text,
            "lxml"
        )

        scripts = soup.find_all(
            "script",
            type="application/ld+json"
        )

        for script in scripts:

            try:

                data = json.loads(
                    script.get_text(
                        strip=True
                    )
                )

            except Exception:

                continue

            if not isinstance(
                data,
                dict
            ):
                continue

            if "headline" not in data:
                continue

            title = data.get(
                "headline",
                ""
            )

            published = data.get(
                "datePublished",
                ""
            )

            author = parse_author(
                data.get("author")
            )

            image_urls, image_captions = (
                parse_images(
                    data.get("image")
                )
            )

            return (

                True,

                {

                    "title":
                        title,

                    "published":
                        published,

                    "author":
                        author,

                    "image_urls":
                        "|".join(
                            image_urls
                        ),

                    "image_captions":
                        "|".join(
                            image_captions
                        ),

                    "image_count":
                        len(
                            image_urls
                        )

                },

                None

            )

        return (
            False,
            None,
            "NO_HEADLINE_JSONLD"
        )

    except Exception as e:

        return (
            False,
            None,
            type(e).__name__
        )

# =====================================================
# LOAD CORPUS
# =====================================================

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

print(
    f"\nArticles loaded: "
    f"{len(df):,}"
)

# =====================================================
# PROCESS
# =====================================================

result_rows = []

failed_rows = []

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = {

        executor.submit(
            extract_metadata,
            row["url"]
        ): row

        for _, row
        in df.iterrows()
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        row = futures[
            future
        ]

        success, metadata, reason = (
            future.result()
        )

        if success:

            row_dict = row.to_dict()

            row_dict.update(
                metadata
            )

            result_rows.append(
                row_dict
            )

        else:

            failed_rows.append({

                "url":
                    row["url"],

                "reason":
                    reason

            })

# =====================================================
# SAVE
# =====================================================

result_df = pd.DataFrame(
    result_rows
)

failed_df = pd.DataFrame(
    failed_rows
)

result_df.to_csv(

    OUTPUT_FILE,

    index=False,

    encoding="utf-8-sig"
)

failed_df.to_csv(

    FAILED_FILE,

    index=False,

    encoding="utf-8-sig"
)

# =====================================================
# DIAGNOSTICS
# =====================================================

print("\n")

print("=" * 60)

print(
    f"Articles processed: "
    f"{len(result_df):,}"
)

print(
    f"Failed: "
    f"{len(failed_df):,}"
)

print(
    f"Articles with images: "
    f"{(result_df['image_count'] > 0).sum():,}"
)

print(
    f"Total image links: "
    f"{result_df['image_count'].sum():,}"
)

print(
    "\nImage count distribution:\n"
)

print(
    result_df[
        "image_count"
    ].describe()
)

if len(failed_df) > 0:

    print(
        "\nFailure reasons:\n"
    )

    print(
        failed_df[
            "reason"
        ].value_counts()
    )

print("\nDONE")

print(
    f"\nSaved:"
)

print(
    OUTPUT_FILE
)

In [ ]:
import pandas as pd

INPUT_FILE = (
    "welt_ukraine_russia_with_metadata.csv"
)

MAPPING_FILE = (
    "welt_image_article_mapping.csv"
)

UNIQUE_FILE = (
    "welt_unique_images.csv"
)

# =====================================================
# LOAD
# =====================================================

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

# =====================================================
# BUILD MAPPING
# =====================================================

mapping_rows = []

for _, row in df.iterrows():

    article_url = row["url"]

    image_urls = str(
        row["image_urls"]
    )

    if image_urls == "nan":
        continue

    for image_url in image_urls.split("|"):

        image_url = image_url.strip()

        if not image_url:
            continue

        mapping_rows.append({

            "article_url":
                article_url,

            "image_url":
                image_url

        })

mapping_df = pd.DataFrame(
    mapping_rows
)

# =====================================================
# UNIQUE IMAGES
# =====================================================

unique_df = (
    mapping_df[
        ["image_url"]
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)

# =====================================================
# SAVE
# =====================================================

mapping_df.to_csv(

    MAPPING_FILE,

    index=False,

    encoding="utf-8-sig"
)

unique_df.to_csv(

    UNIQUE_FILE,

    index=False,

    encoding="utf-8-sig"
)

# =====================================================
# DIAGNOSTICS
# =====================================================

print("\nDONE")

print(
    f"Article-image links: "
    f"{len(mapping_df):,}"
)

print(
    f"Unique images: "
    f"{len(unique_df):,}"
)

print(
    f"Duplicate links removed: "
    f"{len(mapping_df)-len(unique_df):,}"
)

print(
    f"\nSaved:"
)

print(MAPPING_FILE)

print(UNIQUE_FILE)

In [ ]:
import pandas as pd
import re

df = pd.read_csv(
    "welt_unique_images.csv"
)

def extract_image_id(url):

    m = re.search(
        r"/([a-f0-9]{32})/",
        str(url)
    )

    if m:
        return m.group(1)

    return None

df["image_id"] = (
    df["image_url"]
    .apply(extract_image_id)
)

print(
    "URLs:",
    len(df)
)

print(
    "Unique image ids:",
    df["image_id"]
    .nunique()
)

print(
    "Duplicates removed:",
    len(df)
    -
    df["image_id"]
    .nunique()
)

In [ ]:
import pandas as pd
import re

df = pd.read_csv(
    "welt_unique_images.csv"
)

def extract_image_id(url):

    m = re.search(
        r"/([a-f0-9]{32})/",
        str(url)
    )

    if m:
        return m.group(1)

    return None

df["image_id"] = (
    df["image_url"]
    .apply(extract_image_id)
)

deduplicated = (

    df
    .dropna(
        subset=["image_id"]
    )
    .drop_duplicates(
        subset=["image_id"]
    )

)

deduplicated.to_csv(

    "welt_unique_images_deduplicated.csv",

    index=False,

    encoding="utf-8-sig"
)

print(
    len(deduplicated)
)

In [ ]:
import os
import re
import time
import requests
import pandas as pd

from tqdm import tqdm

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

# =====================================================
# SETTINGS
# =====================================================

INPUT_FILE = (
    "welt_unique_images_deduplicated.csv"
)

OUTPUT_DIR = (
    "welt_images"
)

FAILED_FILE = (
    "welt_failed_images.csv"
)

MAX_WORKERS = 10

MAX_RETRIES = 3

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

# =====================================================
# IMAGE ID
# =====================================================

def extract_image_id(url):

    m = re.search(
        r"/([a-f0-9]{32})/",
        str(url)
    )

    if m:
        return m.group(1)

    return None

# =====================================================
# EXTENSION
# =====================================================

def get_extension(url):

    url = str(url).lower()

    for ext in [

        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".gif"

    ]:

        if ext in url:

            return ext

    return ".jpg"

# =====================================================
# DOWNLOAD
# =====================================================

def download_image(row):

    url = row["image_url"]

    image_id = row["image_id"]

    ext = get_extension(
        url
    )

    filename = (
        image_id + ext
    )

    filepath = os.path.join(
        OUTPUT_DIR,
        filename
    )

    if os.path.exists(
        filepath
    ):

        return (
            True,
            "already_exists"
        )

    for attempt in range(
        MAX_RETRIES
    ):

        try:

            response = requests.get(
                url,
                headers=HEADERS,
                timeout=60
            )

            if (
                response.status_code
                == 200
            ):

                with open(
                    filepath,
                    "wb"
                ) as f:

                    f.write(
                        response.content
                    )

                return (
                    True,
                    None
                )

        except Exception:
            pass

        time.sleep(2)

    return (
        False,
        url
    )

# =====================================================
# LOAD
# =====================================================

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

print(
    f"\nImages to download: "
    f"{len(df):,}"
)

# =====================================================
# DOWNLOAD
# =====================================================

failed_urls = []

downloaded = 0

already_exists = 0

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = {

        executor.submit(
            download_image,
            row
        ): row

        for _, row
        in df.iterrows()
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        success, info = (
            future.result()
        )

        if success:

            if info == "already_exists":

                already_exists += 1

            else:

                downloaded += 1

        else:

            failed_urls.append({

                "image_url":
                    info

            })

# =====================================================
# SAVE FAILURES
# =====================================================

failed_df = pd.DataFrame(
    failed_urls
)

failed_df.to_csv(

    FAILED_FILE,

    index=False,

    encoding="utf-8-sig"
)

# =====================================================
# DIAGNOSTICS
# =====================================================

print("\n")

print("=" * 60)

print(
    f"Downloaded: "
    f"{downloaded:,}"
)

print(
    f"Already existed: "
    f"{already_exists:,}"
)

print(
    f"Failed: "
    f"{len(failed_df):,}"
)

print("\nSaved folder:")

print(
    OUTPUT_DIR
)

print("\nFailed file:")

print(
    FAILED_FILE
)

# Spiegel

In [ ]:
import pandas as pd
import requests
import re
import gzip
import io
import time
import random

from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor


# settings

MAIN_SITEMAP = "https://www.spiegel.de/sitemap.xml"

START_YEAR = 2022

MAX_WORKERS = 20

TIMEOUT = 30

SLEEP_RANGE = (0.05, 0.2)


# headers

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    ),

    "Accept-Encoding": "gzip"

}


# fetch xml

def fetch_xml(url):

    try:

        response = requests.get(

            url,

            headers=HEADERS,

            timeout=TIMEOUT

        )

        if response.status_code != 200:
            return None


        content = response.content


        # gzip handling

        if url.endswith(".gz"):

            content = gzip.decompress(content)


        text = content.decode(

            "utf-8",

            errors="ignore"

        )

        return text

    except Exception as e:

        return None


# extract loc urls

def extract_urls(xml_text):

    if not xml_text:
        return []

    return re.findall(

        r"<loc>(.*?)</loc>",

        xml_text

    )


# article detector

def is_article_url(url):

    if not url.startswith(
        "https://www.spiegel.de/"
    ):
        return False

    if "/sitemaps/" in url:
        return False

    if url.endswith(".xml"):
        return False

    if url.endswith(".xml.gz"):
        return False

    if "/video/" in url:
        return False

    if "/podcast/" in url:
        return False

    if "/audio/" in url:
        return False

    if "/fotostrecke/" in url:
        return False

    if "/thema/" in url:
        return False

    if "/schlagzeilen/" in url:
        return False

    if "-a-" in url:
        return True

    return False


# load main sitemap

print("\nloading main sitemap\n")

main_xml = fetch_xml(
    MAIN_SITEMAP
)

main_urls = extract_urls(
    main_xml
)


print("\nurls in sitemap index\n")

print(len(main_urls))


# article shards

article_sitemaps = []


for url in main_urls:

    if "/sitemaps/article/" not in url:
        continue

    match = re.search(

        r"sitemap-(\d{4})-(\d{2})_",

        url

    )

    if not match:
        continue

    year = int(
        match.group(1)
    )

    if year < START_YEAR:
        continue

    article_sitemaps.append(url)


article_sitemaps = sorted(
    set(article_sitemaps)
)


print("\narticle sitemap shards\n")

print(len(article_sitemaps))


print("\nsample shards\n")

for x in article_sitemaps[:20]:

    print(x)


# process shard

def process_sitemap(url):

    xml = fetch_xml(url)

    if not xml:
        return []


    loc_urls = extract_urls(xml)

    article_urls = []


    for loc in loc_urls:

        if is_article_url(loc):

            article_urls.append(loc)


    time.sleep(

        random.uniform(
            *SLEEP_RANGE
        )

    )

    return article_urls


# collect urls

all_article_urls = []


with ThreadPoolExecutor(

    max_workers=MAX_WORKERS

) as executor:

    results = list(

        tqdm(

            executor.map(
                process_sitemap,
                article_sitemaps
            ),

            total=len(article_sitemaps)

        )

    )


for batch in results:

    all_article_urls.extend(batch)


# dedup

all_article_urls = sorted(
    set(all_article_urls)
)


print("\narticle urls found\n")

print(len(all_article_urls))


print("\nsample urls\n")

for url in all_article_urls[:50]:

    print(url)


# save

df = pd.DataFrame({

    "url": all_article_urls

})


df.to_csv(

    "spiegel_article_urls.csv",

    index=False,

    encoding="utf-8-sig"

)


print("\nurls saved\n")

print("spiegel_article_urls.csv")


print("\nfinished")

In [ ]:
import pandas as pd
import aiohttp
import asyncio
import csv
import re
import json
import random

from bs4 import BeautifulSoup
from tqdm.asyncio import tqdm


# settings

INPUT_URLS = "spiegel_article_urls.csv"

OUTPUT_FILE = "spiegel_corpus.csv"

ERROR_FILE = "spiegel_errors.csv"

CONCURRENT_REQUESTS = 30

TIMEOUT = 30

SLEEP_RANGE = (0.05, 0.2)


# headers

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )

}


# load urls

urls_df = pd.read_csv(INPUT_URLS)

if "article_url" in urls_df.columns:

    article_urls = (
        urls_df["article_url"]
        .dropna()
        .unique()
        .tolist()
    )

else:

    article_urls = (
        urls_df.iloc[:, 0]
        .dropna()
        .unique()
        .tolist()
    )


print("\narticle urls\n")

print(len(article_urls))


# csv locks

csv_lock = asyncio.Lock()

error_lock = asyncio.Lock()


# init csv

columns = [

    "article_url",
    "article_id",
    "section",
    "published_time",
    "title",
    "article_text",
    "image_urls",
    "image_count"

]

with open(

    OUTPUT_FILE,

    "w",

    newline="",

    encoding="utf-8-sig"

) as f:

    writer = csv.DictWriter(

        f,

        fieldnames=columns

    )

    writer.writeheader()


# init errors

with open(

    ERROR_FILE,

    "w",

    newline="",

    encoding="utf-8-sig"

) as f:

    writer = csv.writer(f)

    writer.writerow([

        "article_url",
        "error"

    ])


# extract ld json

def extract_ld_json(soup):

    scripts = soup.find_all(

        "script",

        type="application/ld+json"

    )

    for script in scripts:

        try:

            data = json.loads(

                script.string

            )

            if isinstance(data, dict):

                if data.get("@type") in [

                    "NewsArticle",
                    "Article"

                ]:

                    return data

        except:
            pass

    return {}


# async parser

async def parse_article(

    session,
    semaphore,
    url

):

    async with semaphore:

        try:

            await asyncio.sleep(

                random.uniform(
                    *SLEEP_RANGE
                )

            )


            async with session.get(

                url,

                timeout=TIMEOUT

            ) as response:

                response.raise_for_status()

                html = await response.text()


            soup = BeautifulSoup(

                html,

                "html.parser"

            )


            ld = extract_ld_json(soup)


            # title

            title = ""

            if ld.get("headline"):

                title = ld["headline"]

            elif soup.title:

                title = soup.title.text.strip()


            # published time

            published_time = (

                ld.get(
                    "datePublished",
                    ""
                )

            )


            # section

            section = ""

            try:

                section = (

                    url
                    .replace(
                        "https://www.spiegel.de/",
                        ""
                    )
                    .split("/")[0]

                )

            except:
                pass


            # text

            paragraphs = soup.find_all("p")

            text_parts = []

            for p in paragraphs:

                txt = p.get_text(

                    " ",

                    strip=True

                )

                if len(txt) > 50:

                    text_parts.append(txt)


            article_text = "\n".join(text_parts)


            # images

            image_urls = set()


            for img in soup.find_all("img"):

                src = (

                    img.get("src")

                    or

                    img.get("data-src")

                )

                if not src:
                    continue

                if src.startswith("http"):

                    image_urls.add(src)


            image_urls = list(image_urls)


            # article id

            article_id = ""

            match = re.search(

                r"a-([a-zA-Z0-9\-]+)\.html",

                url

            )

            if match:

                article_id = match.group(1)


            row = {

                "article_url": url,

                "article_id": article_id,

                "section": section,

                "published_time": published_time,

                "title": title,

                "article_text": article_text,

                "image_urls": json.dumps(
                    image_urls,
                    ensure_ascii=False
                ),

                "image_count": len(image_urls)

            }


            # safe csv write

            async with csv_lock:

                with open(

                    OUTPUT_FILE,

                    "a",

                    newline="",

                    encoding="utf-8-sig"

                ) as f:

                    writer = csv.DictWriter(

                        f,

                        fieldnames=columns

                    )

                    writer.writerow(row)


        except Exception as e:

            print("\nerror\n")

            print(url)

            print(e)


            async with error_lock:

                with open(

                    ERROR_FILE,

                    "a",

                    newline="",

                    encoding="utf-8-sig"

                ) as f:

                    writer = csv.writer(f)

                    writer.writerow([

                        url,
                        str(e)

                    ])


# main

async def main():

    semaphore = asyncio.Semaphore(

        CONCURRENT_REQUESTS

    )


    async with aiohttp.ClientSession(

        headers=HEADERS

    ) as session:


        tasks = [

            parse_article(

                session,
                semaphore,
                url

            )

            for url in article_urls

        ]


        await asyncio.gather(*tasks)


# run

await main()

print("\nfinished")

In [ ]:
df = pd.read_csv("spiegel_corpus.csv")

In [ ]:
df

In [ ]:
import pandas as pd
import aiohttp
import asyncio
import os
import re
import json
import random

from bs4 import BeautifulSoup
from tqdm.asyncio import tqdm


# settings

INPUT_URLS = "spiegel_article_urls.csv"

OUTPUT_FILE = "spiegel_corpus.csv"

FAILED_FILE = "spiegel_failed_urls.csv"

CONCURRENT_REQUESTS = 40

TIMEOUT = 30

SAVE_EVERY = 100

RESUME = True


# headers

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )

}


# load urls

urls_df = pd.read_csv(INPUT_URLS)

url_column = (
    "article_url"
    if "article_url" in urls_df.columns
    else urls_df.columns[0]
)

urls = (
    urls_df[url_column]
    .dropna()
    .unique()
    .tolist()
)


print("\narticle urls\n")

print(len(urls))


# resume

already_done = set()

if RESUME and os.path.exists(OUTPUT_FILE):

    old_df = pd.read_csv(OUTPUT_FILE)

    if "article_url" in old_df.columns:

        already_done = set(

            old_df["article_url"]
            .dropna()
            .tolist()

        )

    print("\nalready processed\n")

    print(len(already_done))

    urls = [

        u for u in urls

        if u not in already_done

    ]


print("\nremaining urls\n")

print(len(urls))


# create files

if not os.path.exists(OUTPUT_FILE):

    pd.DataFrame(columns=[

        "article_url",
        "section",
        "published_time",
        "title",
        "article_text",
        "image_urls",
        "image_count"

    ]).to_csv(

        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"

    )


if not os.path.exists(FAILED_FILE):

    pd.DataFrame(columns=[

        "article_url",
        "error"

    ]).to_csv(

        FAILED_FILE,
        index=False,
        encoding="utf-8-sig"

    )


# locks

csv_lock = asyncio.Lock()

failed_lock = asyncio.Lock()


# buffers

buffer = []

failed_buffer = []


# extract section

def extract_section(url):

    try:

        path = (
            url
            .replace("https://www.spiegel.de/", "")
            .split("/")
        )

        if len(path) > 0:
            return path[0]

        return None

    except:
        return None


# extract metadata

async def extract_article(

    session,
    url

):

    global buffer
    global failed_buffer

    try:

        timeout = aiohttp.ClientTimeout(
            total=TIMEOUT
        )

        async with session.get(

            url,

            headers=HEADERS,

            timeout=timeout

        ) as response:

            html = await response.text()


        soup = BeautifulSoup(
            html,
            "html.parser"
        )


        # title

        title = None

        if soup.title:

            title = soup.title.text.strip()


        og_title = soup.find(

            "meta",
            property="og:title"

        )

        if og_title:

            title = og_title.get(
                "content",
                title
            )


        # article text

        paragraphs = soup.find_all("p")

        article_text = " ".join([

            p.get_text(
                " ",
                strip=True
            )

            for p in paragraphs

        ])


        # images

        image_urls = set()


        for img in soup.find_all("img"):

            src = (
                img.get("src")
                or img.get("data-src")
            )

            if not src:
                continue

            if src.startswith("//"):
                src = "https:" + src

            if not src.startswith("http"):
                continue

            if any(

                x in src.lower()

                for x in [

                    ".jpg",
                    ".jpeg",
                    ".png",
                    ".webp"

                ]

            ):

                image_urls.add(src)


        # published time

        published_time = None

        meta_time = soup.find(

            "meta",

            attrs={

                "property": "article:published_time"

            }

        )

        if meta_time:

            published_time = meta_time.get(
                "content"
            )


        # section

        section = extract_section(url)


        data = {

            "article_url": url,

            "section": section,

            "published_time": published_time,

            "title": title,

            "article_text": article_text,

            "image_urls": json.dumps(
                list(image_urls),
                ensure_ascii=False
            ),

            "image_count": len(image_urls)

        }


        buffer.append(data)


        # save chunk

        if len(buffer) >= SAVE_EVERY:

            async with csv_lock:

                pd.DataFrame(
                    buffer
                ).to_csv(

                    OUTPUT_FILE,

                    mode="a",

                    header=False,

                    index=False,

                    encoding="utf-8-sig"

                )

                print(

                    f"\nsaved rows: {len(buffer)}\n"

                )

                buffer = []


        await asyncio.sleep(

            random.uniform(
                0.05,
                0.2
            )

        )


    except Exception as e:

        print("\nerror\n")

        print(url)

        print(str(e))


        failed_buffer.append({

            "article_url": url,

            "error": str(e)

        })


        if len(failed_buffer) >= 20:

            async with failed_lock:

                pd.DataFrame(
                    failed_buffer
                ).to_csv(

                    FAILED_FILE,

                    mode="a",

                    header=False,

                    index=False,

                    encoding="utf-8-sig"

                )

                failed_buffer.clear()


# worker

async def worker(

    session,
    semaphore,
    url

):

    async with semaphore:

        await extract_article(
            session,
            url
        )


# main

async def main():

    semaphore = asyncio.Semaphore(
        CONCURRENT_REQUESTS
    )

    connector = aiohttp.TCPConnector(

        limit=CONCURRENT_REQUESTS

    )

    timeout = aiohttp.ClientTimeout(
        total=TIMEOUT
    )


    async with aiohttp.ClientSession(

        connector=connector,
        timeout=timeout

    ) as session:

        tasks = [

            worker(

                session,
                semaphore,
                url

            )

            for url in urls

        ]


        for future in tqdm.as_completed(tasks):

            await future


    # final save

    global buffer
    global failed_buffer


    if len(buffer) > 0:

        async with csv_lock:

            pd.DataFrame(
                buffer
            ).to_csv(

                OUTPUT_FILE,

                mode="a",

                header=False,

                index=False,

                encoding="utf-8-sig"

            )


    if len(failed_buffer) > 0:

        async with failed_lock:

            pd.DataFrame(
                failed_buffer
            ).to_csv(

                FAILED_FILE,

                mode="a",

                header=False,

                index=False,

                encoding="utf-8-sig"

            )


# run in jupyter

await main()


print("\nfinished")

In [ ]:
import pandas as pd
import re


# settings

input_file = "spiegel_corpus.csv"

output_file = "spiegel_ukraine_russia_high_recall.csv"


# load corpus

df = pd.read_csv(input_file)


print("\ninput corpus\n")

print(len(df))


# anchor keywords

anchor_patterns = [

    # ukraine

    r"ukrain\w*",
    r"ukrainisch\w*",
    r"ukrainer\w*",

    # russia

    r"russland\w*",
    r"russisch\w*",
    r"russ\w*",
    r"russen\w*",

    # putin

    r"putin\w*",

    # zelensky

    r"selensky\w*",
    r"zelensky\w*",
    r"selens\w*",
    r"zelens\w*"

]


# war / geopolitical context

context_patterns = [

    r"krieg\w*",
    r"angriff\w*",
    r"invasion\w*",
    r"offensive\w*",

    r"nato\w*",
    r"waffen\w*",
    r"raket\w*",
    r"drohn\w*",
    r"panzer\w*",

    r"donbass\w*",
    r"donezk\w*",
    r"luhansk\w*",
    r"krim\w*",
    r"crimea\w*",

    r"kiew\w*",
    r"kyiv\w*",
    r"moskau\w*",

    r"kreml\w*",

    r"gaslieferung\w*",
    r"nord\s*stream\w*",
    r"energiekrise\w*"

]


# compile regex

anchor_regex = re.compile(

    "|".join(anchor_patterns),

    flags=re.IGNORECASE

)

context_regex = re.compile(

    "|".join(context_patterns),

    flags=re.IGNORECASE

)


# combine text

df["combined_text"] = (

    df["title"]
    .fillna("")
    + " "
    + df["article_text"]
    .fillna("")

)


# filtering

filtered = df[

    df["combined_text"]
    .str.contains(
        anchor_regex,
        na=False
    )

    &

    df["combined_text"]
    .str.contains(
        context_regex,
        na=False
    )

]


# dedup

filtered = filtered.drop_duplicates(

    subset=["article_url"]

)


# diagnostics

print("\nfiltered corpus\n")

print(len(filtered))


print("\nsection distribution\n")

print(

    filtered["section"]
    .value_counts()
    .head(30)

)


# years

filtered["published_time"] = pd.to_datetime(

    filtered["published_time"],

    errors="coerce",

    utc=True

)


print("\nyear distribution\n")

print(

    filtered["published_time"]
    .dt.year
    .value_counts()
    .sort_index()

)


print("\nimage count distribution\n")

print(

    filtered["image_count"]
    .describe()

)


# articles with images

filtered["has_images"] = (

    filtered["image_count"] > 0

)


print("\narticles with images\n")

print(

    filtered["has_images"]
    .value_counts()

)


# save

filtered.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nfiltered corpus saved\n")

print(output_file)


print("\nfinished")

In [ ]:
import pandas as pd
import requests

from bs4 import BeautifulSoup


# settings

input_file = "spiegel_ukraine_russia_high_recall.csv"

sample_size = 10

timeout = 30


# headers

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )

}


# load dataset

df = pd.read_csv(input_file)


sample = df.sample(

    sample_size,

    random_state=42

)


# test extraction

for idx, row in sample.iterrows():

    url = row["article_url"]


    print("\n" + "=" * 80)

    print("\narticle\n")

    print(url)


    try:

        response = requests.get(

            url,

            headers=headers,

            timeout=timeout

        )


        soup = BeautifulSoup(

            response.text,

            "html.parser"

        )


        og_image = soup.find(

            "meta",

            property="og:image"

        )


        if og_image:

            print("\nog:image\n")

            print(

                og_image.get(
                    "content"
                )

            )

        else:

            print("\nno og:image found\n")


    except Exception as e:

        print("\nerror\n")

        print(e)

### lost some urls, need to find them

In [ ]:
import pandas as pd
import aiohttp
import asyncio
import nest_asyncio
import re

from tqdm import tqdm


# notebook async fix

nest_asyncio.apply()


# settings

input_file = "spiegel_ukraine_russia_high_recall.csv"

output_file = "spiegel_ukraine_russia_fixed_images.csv"

concurrency = 15

timeout_seconds = 30


# headers

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )

}


# load

df = pd.read_csv(input_file)


print("\ninput rows\n")

print(len(df))


# VERY IMPORTANT

df["image_count"] = pd.to_numeric(

    df["image_count"],

    errors="coerce"

)


# proper broken detection

broken_mask = (

    df["image_urls"].isna()

    |

    df["image_urls"].astype(str).isin([

        "",

        "[]",

        "nan",

        "None"

    ])

    |

    df["image_count"].isna()

    |

    (df["image_count"] <= 0)

)


broken = df[
    broken_mask
].copy()


print("\nbroken rows\n")

print(len(broken))


# async semaphore

semaphore = asyncio.Semaphore(
    concurrency
)


# regex

og_pattern = re.compile(

    r'<meta[^>]+property=["\']og:image["\'][^>]+content=["\']([^"\']+)["\']',

    re.IGNORECASE

)


twitter_pattern = re.compile(

    r'<meta[^>]+name=["\']twitter:image["\'][^>]+content=["\']([^"\']+)["\']',

    re.IGNORECASE

)


# extractor

async def extract_main_image(

    session,
    row

):

    url = row["article_url"]


    async with semaphore:

        try:

            async with session.get(

                url,

                timeout=timeout_seconds

            ) as response:

                html = await response.text(

                    errors="ignore"

                )


            main_image = None


            # og:image

            og_match = og_pattern.search(
                html
            )


            if og_match:

                main_image = og_match.group(1)


            # twitter fallback

            if not main_image:

                twitter_match = twitter_pattern.search(
                    html
                )


                if twitter_match:

                    main_image = twitter_match.group(1)


            # update

            if main_image:

                row["image_urls"] = str(
                    [main_image]
                )

                row["image_count"] = 1


            return row


        except Exception as e:

            print("\nerror\n")

            print(url)

            print(e)

            return row


# async processing

async def process_all():

    connector = aiohttp.TCPConnector(

        limit=concurrency,

        ssl=False

    )


    timeout = aiohttp.ClientTimeout(

        total=timeout_seconds

    )


    async with aiohttp.ClientSession(

        headers=headers,

        connector=connector,

        timeout=timeout

    ) as session:


        tasks = [

            extract_main_image(

                session,
                row.copy()

            )

            for idx, row in broken.iterrows()

        ]


        results = []


        for coro in tqdm(

            asyncio.as_completed(tasks),

            total=len(tasks)

        ):

            result = await coro

            results.append(result)


    return results


# run

fixed_rows = asyncio.get_event_loop().run_until_complete(

    process_all()

)


# merge back

fixed_df = df.copy()


for row in fixed_rows:

    fixed_df.loc[

        fixed_df["article_url"]

        ==

        row["article_url"]

    ] = row


# diagnostics

remaining = fixed_df[

    fixed_df["image_urls"].isna()

    |

    fixed_df["image_urls"].astype(str).isin([

        "",

        "[]",

        "nan",

        "None"

    ])

    |

    fixed_df["image_count"].isna()

    |

    (fixed_df["image_count"] <= 0)

]


print("\nremaining broken rows\n")

print(len(remaining))


# save

fixed_df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nfixed dataset saved\n")

print(output_file)


print("\nfinished")

In [ ]:
# merge back

fixed_df = df.copy()


# convert rows to dataframe

fixed_rows_df = pd.DataFrame(
    fixed_rows
)


# remove duplicates

fixed_rows_df = fixed_rows_df.drop_duplicates(

    subset=["article_url"]

)


# set index

fixed_df = fixed_df.set_index(
    "article_url"
)

fixed_rows_df = fixed_rows_df.set_index(
    "article_url"
)


# update only repaired rows

fixed_df.update(
    fixed_rows_df
)


# restore index

fixed_df = fixed_df.reset_index()


# diagnostics

remaining = fixed_df[

    fixed_df["image_urls"].isna()

    |

    fixed_df["image_urls"].astype(str).isin([

        "",

        "[]",

        "nan",

        "None"

    ])

    |

    fixed_df["image_count"].isna()

    |

    (fixed_df["image_count"] <= 0)

]


print("\nremaining broken rows\n")

print(len(remaining))


# save

fixed_df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nfixed dataset saved\n")

print(output_file)


print("\nfinished")

### diagnostics

In [ ]:
import pandas as pd
import ast


file = "spiegel_ukraine_russia_fixed_images.csv"


df = pd.read_csv(file)


print("\nrows\n")

print(len(df))


# missing image_urls

missing_urls = df[
    df["image_urls"].isna()
]


print("\nmissing image_urls\n")

print(len(missing_urls))


# missing image_count

missing_count = df[
    df["image_count"].isna()
]


print("\nmissing image_count\n")

print(len(missing_count))


# zero image_count

zero_count = df[
    df["image_count"] == 0
]


print("\nzero image_count\n")

print(len(zero_count))


# invalid image urls

invalid = []


for idx, row in df.iterrows():

    image_urls = row["image_urls"]


    if pd.isna(image_urls):
        continue


    try:

        parsed = ast.literal_eval(
            image_urls
        )


        if not isinstance(parsed, list):

            invalid.append(
                row["article_url"]
            )

            continue


        if len(parsed) == 0:

            invalid.append(
                row["article_url"]
            )

            continue


        first = parsed[0]


        if not isinstance(first, str):

            invalid.append(
                row["article_url"]
            )

            continue


        if not first.startswith("http"):

            invalid.append(
                row["article_url"]
            )

            continue


    except:

        invalid.append(
            row["article_url"]
        )


print("\ninvalid image urls\n")

print(len(invalid))


print("\nfinished")

In [ ]:
sample = df.sample(

    20,

    random_state=42

)


for idx, row in sample.iterrows():

    print("\nARTICLE\n")

    print(row["article_url"])


    print("\nIMAGE\n")

    print(row["image_urls"])

### clean noisy image urls leave only main image

In [ ]:
import pandas as pd
import ast
import re


# settings

input_file = "spiegel_ukraine_russia_fixed_images.csv"

output_file = "spiegel_clean_main_images.csv"


# load

df = pd.read_csv(input_file)


print("\nrows\n")

print(len(df))


# helper

def select_main_image(image_urls):

    # missing

    if pd.isna(image_urls):

        return None


    # parse list

    try:

        urls = ast.literal_eval(
            image_urls
        )

    except:

        return None


    if not isinstance(urls, list):

        return None


    if len(urls) == 0:

        return None


    # keep strings only

    urls = [

        u for u in urls

        if isinstance(u, str)

    ]


    if len(urls) == 0:

        return None


    # remove obvious junk

    bad_terms = [

        "logo",
        "icon",
        "avatar",
        "tracking",
        "pixel",
        "button",
        "newsletter",
        "facebook",
        "twitter",
        "instagram",
        "whatsapp",
        "sprite",
        "favicon",
        "author",
        "social"

    ]


    clean_urls = []


    for url in urls:

        lower = url.lower()


        if any(

            bad in lower

            for bad in bad_terms

        ):

            continue


        clean_urls.append(url)


    # if everything removed

    if len(clean_urls) == 0:

        clean_urls = urls


    # prefer spiegel cdn

    spiegel_urls = [

        u for u in clean_urls

        if "cdn.prod.www.spiegel.de" in u

    ]


    if len(spiegel_urls) > 0:

        clean_urls = spiegel_urls


    # scoring

    scored = []


    for url in clean_urls:

        score = 0


        lower = url.lower()


        # prefer image formats

        if ".jpg" in lower:
            score += 5

        if ".jpeg" in lower:
            score += 5

        if ".png" in lower:
            score += 3

        if ".webp" in lower:
            score += 3


        # resolution hints

        resolution_matches = re.findall(

            r"(\d{3,4})x(\d{3,4})",

            lower

        )


        if len(resolution_matches) > 0:

            try:

                w, h = resolution_matches[0]

                score += int(w) * int(h)

            except:
                pass


        # prefer long urls

        score += len(url)


        scored.append(

            (url, score)

        )


    # sort best first

    scored = sorted(

        scored,

        key=lambda x: x[1],

        reverse=True

    )


    # best image

    return scored[0][0]


# process dataset

main_images = []


for idx, row in df.iterrows():

    main_image = select_main_image(

        row["image_urls"]

    )


    row["main_image"] = main_image


    if main_image:

        row["image_urls"] = str(
            [main_image]
        )

        row["image_count"] = 1

        row["has_main_image"] = True

    else:

        row["image_urls"] = str([])

        row["image_count"] = 0

        row["has_main_image"] = False


    main_images.append(row)


# final dataset

final_df = pd.DataFrame(
    main_images
)


# diagnostics

print("\nmain image stats\n")

print(

    final_df["has_main_image"]
    .value_counts()

)


print("\nremaining missing images\n")

print(

    (
        final_df["image_count"] == 0
    ).sum()

)


# sample

print("\nsample rows\n")


sample = final_df.sample(
    10,
    random_state=42
)


for idx, row in sample.iterrows():

    print("\narticle\n")

    print(row["article_url"])


    print("\nmain image\n")

    print(row["main_image"])


# save

final_df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nclean dataset saved\n")

print(output_file)


print("\nfinished")

In [ ]:
import pandas as pd
import requests
import os
import ast
import hashlib
import time
import random

from tqdm import tqdm
from urllib.parse import urlparse


# settings

input_file = "spiegel_clean_main_images.csv"

output_dir = "spiegel_main_images"

download_log = "spiegel_downloaded_images.csv"

failed_log = "spiegel_failed_images.csv"


# download settings

sleep_range = (0.05, 0.2)

timeout = 30

resume = True


# headers

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )

}


# folders

os.makedirs(

    output_dir,

    exist_ok=True

)


# load dataset

df = pd.read_csv(input_file)


print("\ninput rows\n")

print(len(df))


# parse image urls

def parse_urls(x):

    if pd.isna(x):

        return []


    try:

        parsed = ast.literal_eval(x)

        if isinstance(parsed, list):

            return parsed

        return []

    except:

        return []


df["image_urls"] = df["image_urls"].apply(
    parse_urls
)


# keep only rows with images

df = df[

    df["image_urls"].apply(len) > 0

].copy()


print("\nrows with images\n")

print(len(df))


# resume logic

already_downloaded = set()


if resume and os.path.exists(download_log):

    old = pd.read_csv(download_log)


    if "image_url" in old.columns:

        already_downloaded = set(

            old["image_url"]
            .dropna()
            .tolist()

        )


print("\nalready downloaded\n")

print(len(already_downloaded))


# logs

downloaded = []

failed = []


# download loop

for idx, row in tqdm(

    df.iterrows(),

    total=len(df)

):

    article_url = row["article_url"]

    section = row.get(

        "section",

        "unknown"

    )


    image_urls = row["image_urls"]


    # only one main image

    image_url = image_urls[0]


    # skip existing

    if image_url in already_downloaded:

        continue


    try:

        # section folder

        section_folder = os.path.join(

            output_dir,

            str(section)

        )


        os.makedirs(

            section_folder,

            exist_ok=True

        )


        # extension

        parsed = urlparse(image_url)

        ext = os.path.splitext(
            parsed.path
        )[1].lower()


        if ext not in [

            ".jpg",
            ".jpeg",
            ".png",
            ".webp"

        ]:

            ext = ".jpg"


        # filename

        image_hash = hashlib.md5(

            image_url.encode()

        ).hexdigest()


        filename = f"{image_hash}{ext}"


        filepath = os.path.join(

            section_folder,

            filename

        )


        # skip local existing

        if os.path.exists(filepath):

            continue


        # request

        response = requests.get(

            image_url,

            headers=headers,

            timeout=timeout,

            stream=True

        )


        response.raise_for_status()


        # save image

        with open(filepath, "wb") as f:

            for chunk in response.iter_content(
                8192
            ):

                f.write(chunk)


        # success log

        downloaded.append({

            "article_url": article_url,

            "section": section,

            "image_url": image_url,

            "filepath": filepath,

            "status": "downloaded"

        })


        # periodic save

        if len(downloaded) % 500 == 0:

            pd.DataFrame(downloaded).to_csv(

                download_log,

                mode="a",

                header=not os.path.exists(download_log),

                index=False,

                encoding="utf-8-sig"

            )

            downloaded = []


        # polite crawling

        time.sleep(

            random.uniform(
                *sleep_range
            )

        )


    except Exception as e:

        failed.append({

            "article_url": article_url,

            "image_url": image_url,

            "error": str(e)

        })


        print("\nerror\n")

        print(image_url)

        print(e)


        # periodic failed save

        if len(failed) % 100 == 0:

            pd.DataFrame(failed).to_csv(

                failed_log,

                mode="a",

                header=not os.path.exists(failed_log),

                index=False,

                encoding="utf-8-sig"

            )

            failed = []


# final save

if len(downloaded) > 0:

    pd.DataFrame(downloaded).to_csv(

        download_log,

        mode="a",

        header=not os.path.exists(download_log),

        index=False,

        encoding="utf-8-sig"

    )


if len(failed) > 0:

    pd.DataFrame(failed).to_csv(

        failed_log,

        mode="a",

        header=not os.path.exists(failed_log),

        index=False,

        encoding="utf-8-sig"

    )


print("\nfinished\n")

print("download log")

print(download_log)

print("\nfailed log")

print(failed_log)

In [ ]:
import pandas as pd


downloaded = pd.read_csv(
    "spiegel_downloaded_images.csv"
)

failed = pd.read_csv(
    "spiegel_failed_images.csv"
)


print("\ndownloaded\n")

print(len(downloaded))


print("\nfailed\n")

print(len(failed))


total = len(downloaded) + len(failed)


print("\ntotal processed\n")

print(total)

In [ ]:
import pandas as pd
import ast


df = pd.read_csv(
    "spiegel_clean_main_images.csv"
)


# parse

def parse_urls(x):

    try:
        return ast.literal_eval(x)

    except:
        return []


df["image_urls"] = df["image_urls"].apply(
    parse_urls
)


# first image only

df["main_image"] = df["image_urls"].apply(

    lambda x: x[0]

    if len(x) > 0

    else None

)


unique_images = df["main_image"].nunique()


print("\narticles\n")

print(len(df))


print("\nunique image urls\n")

print(unique_images)

In [ ]:
import pandas as pd
import ast


df = pd.read_csv(
    "spiegel_clean_main_images.csv"
)


def parse_urls(x):

    try:
        return ast.literal_eval(x)

    except:
        return []


df["image_urls"] = df["image_urls"].apply(
    parse_urls
)


df["main_image"] = df["image_urls"].apply(

    lambda x: x[0]

    if len(x) > 0

    else None

)


# how many rows lost image completely

missing = df["main_image"].isna().sum()


print("\nmissing main images\n")

print(missing)


print("\nrows total\n")

print(len(df))

In [ ]:
for section in [

    "ausland",
    "politik",
    "sport",
    "kultur"

]:

    subset = df[
        df["section"] == section
    ]


    sample = subset.sample(5)


    print("\nSECTION\n")

    print(section)


    for idx, row in sample.iterrows():

        print(row["article_url"])

        print(row["main_image"])

        print()

## image url download retry

In [ ]:
import pandas as pd
import requests
import re

from bs4 import BeautifulSoup
from tqdm import tqdm


# ==================================================
# settings
# ==================================================

input_file = "spiegel_ukraine_russia_high_recall.csv"

output_file = "spiegel_clean_article_images.csv"

timeout = 30


# ==================================================
# headers
# ==================================================

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )

}


# ==================================================
# load dataset
# ==================================================

df = pd.read_csv(input_file)


print("\nrows\n")

print(len(df))


# ==================================================
# helper
# ==================================================

def clean_url(url):

    if not url:
        return None


    if not isinstance(url, str):
        return None


    lower = url.lower()


    # must look like image

    if not any(

        ext in lower

        for ext in [

            ".jpg",
            ".jpeg",
            ".png",
            ".webp",
            ".avif"

        ]

    ):

        return None


    # obvious junk

    bad_terms = [

        "logo",
        "icon",
        "avatar",
        "tracking",
        "pixel",
        "sprite",
        "favicon",
        "ads",
        "banner",
        "button"

    ]


    if any(

        bad in lower

        for bad in bad_terms

    ):

        return None


    # relative urls

    if lower.startswith("//"):

        url = "https:" + url


    elif lower.startswith("/"):

        url = "https://www.spiegel.de" + url


    return url


# ==================================================
# extraction
# ==================================================

def extract_article_images(soup):

    images = []


    # ----------------------------------------------
    # article containers
    # ----------------------------------------------

    containers = []


    article_tag = soup.find("article")


    if article_tag:

        containers.append(article_tag)


    # additional likely containers

    for cls in [

        "RichText",
        "Article",
        "article-section",
        "js-article-body"

    ]:

        found = soup.find_all(

            class_=re.compile(
                cls,
                re.I
            )

        )

        containers.extend(found)


    # fallback

    if len(containers) == 0:

        containers = [soup]


    # ----------------------------------------------
    # extract image candidates
    # ----------------------------------------------

    for container in containers:

        imgs = container.find_all("img")


        for img in imgs:

            candidates = []


            # src

            src = img.get("src")

            if src:

                candidates.append(src)


            # data-src

            data_src = img.get(
                "data-src"
            )

            if data_src:

                candidates.append(data_src)


            # srcset

            srcset = img.get(
                "srcset"
            )


            if srcset:

                parts = srcset.split(",")


                for part in parts:

                    piece = (

                        part.strip()
                        .split(" ")[0]

                    )

                    candidates.append(piece)


            # clean candidates

            for candidate in candidates:

                cleaned = clean_url(
                    candidate
                )


                if cleaned:

                    images.append(cleaned)


    # ----------------------------------------------
    # deduplicate resolutions
    # ----------------------------------------------

    best_versions = {}


    for img in images:

        lower = img.lower()


        # extract image uuid

        match = re.search(

            r'/images/([a-f0-9\-]+)',

            lower

        )


        if match:

            image_id = match.group(1)

        else:

            image_id = lower


        # score quality

        score = 0


        # width

        width_match = re.search(

            r'_w(\d+)',

            lower

        )


        if width_match:

            try:

                score += int(
                    width_match.group(1)
                )

            except:
                pass


        # prefer jpg/jpeg

        if ".jpg" in lower:
            score += 1000

        if ".jpeg" in lower:
            score += 1000


        # keep best version only

        if image_id not in best_versions:

            best_versions[image_id] = (

                img,
                score

            )

        else:

            old_img, old_score = best_versions[
                image_id
            ]


            if score > old_score:

                best_versions[image_id] = (

                    img,
                    score

                )


    # ----------------------------------------------
    # final images
    # ----------------------------------------------

    unique = [

        value[0]

        for value in best_versions.values()

    ]


    return unique


# ==================================================
# process dataset
# ==================================================

results = []


for idx, row in tqdm(

    df.iterrows(),

    total=len(df)

):

    article_url = row["article_url"]


    try:

        response = requests.get(

            article_url,

            headers=headers,

            timeout=timeout

        )


        soup = BeautifulSoup(

            response.text,

            "html.parser"

        )


        images = extract_article_images(
            soup
        )


        row["image_urls"] = str(
            images
        )

        row["image_count"] = len(
            images
        )

        row["has_images"] = (

            len(images) > 0

        )


        results.append(row)


    except Exception as e:

        print("\nerror\n")

        print(article_url)

        print(e)


        row["image_urls"] = str([])

        row["image_count"] = 0

        row["has_images"] = False


        results.append(row)


# ==================================================
# final dataframe
# ==================================================

final_df = pd.DataFrame(
    results
)


# ==================================================
# diagnostics
# ==================================================

print("\narticles with images\n")

print(

    final_df["has_images"]
    .value_counts()

)


print("\nimage count stats\n")

print(

    final_df["image_count"]
    .describe()

)


print("\nrows without images\n")

print(

    (
        final_df["image_count"] == 0
    ).sum()

)


# ==================================================
# sample
# ==================================================

print("\nrandom sample\n")


sample = final_df.sample(10)


for idx, row in sample.iterrows():

    print("\n====================\n")

    print(row["article_url"])


    print("\nimage count\n")

    print(row["image_count"])


    try:

        urls = eval(
            row["image_urls"]
        )

    except:

        urls = []


    for url in urls[:10]:

        print(url)


# ==================================================
# save
# ==================================================

final_df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nclean dataset saved\n")

print(output_file)


print("\nfinished")

In [ ]:
import pandas as pd
import ast


# ==================================================
# load
# ==================================================

file = "spiegel_clean_article_images.csv"


df = pd.read_csv(file)


print("\nrows\n")

print(len(df))


# ==================================================
# missing image_urls
# ==================================================

missing_urls = df[

    df["image_urls"].isna()

]


print("\nmissing image_urls\n")

print(len(missing_urls))


# ==================================================
# zero image_count
# ==================================================

zero_images = df[

    df["image_count"] == 0

]


print("\nrows with zero images\n")

print(len(zero_images))


# ==================================================
# invalid image lists
# ==================================================

invalid_rows = []


for idx, row in df.iterrows():

    image_urls = row["image_urls"]


    if pd.isna(image_urls):

        invalid_rows.append(
            row["article_url"]
        )

        continue


    try:

        parsed = ast.literal_eval(
            image_urls
        )


        if not isinstance(parsed, list):

            invalid_rows.append(
                row["article_url"]
            )

            continue


        # check urls

        for url in parsed:

            if not isinstance(url, str):

                invalid_rows.append(
                    row["article_url"]
                )

                break


            if not url.startswith("http"):

                invalid_rows.append(
                    row["article_url"]
                )

                break


    except:

        invalid_rows.append(
            row["article_url"]
        )


print("\ninvalid image rows\n")

print(len(invalid_rows))


# ==================================================
# duplicates
# ==================================================

duplicate_articles = df[

    df["article_url"]
    .duplicated()

]


print("\nduplicate article urls\n")

print(len(duplicate_articles))


# ==================================================
# image count stats
# ==================================================

print("\nimage count stats\n")

print(

    df["image_count"]
    .describe()

)


# ==================================================
# articles with images
# ==================================================

print("\narticles with images\n")

print(

    df["has_images"]
    .value_counts()

)


# ==================================================
# unique images
# ==================================================

all_images = []


for idx, row in df.iterrows():

    try:

        parsed = ast.literal_eval(

            row["image_urls"]

        )


        all_images.extend(parsed)

    except:
        pass


unique_images = len(

    set(all_images)

)


print("\nunique images\n")

print(unique_images)


# ==================================================
# repeated images
# ==================================================

image_counts = pd.Series(

    all_images

).value_counts()


print("\nmost repeated images\n")

print(

    image_counts.head(20)

)


# ==================================================
# random sample
# ==================================================

print("\nrandom sample\n")


sample = df.sample(15)


for idx, row in sample.iterrows():

    print("\n====================\n")

    print("ARTICLE")

    print(row["article_url"])


    print("\nIMAGE COUNT")

    print(row["image_count"])


    try:

        urls = ast.literal_eval(
            row["image_urls"]
        )

    except:

        urls = []


    print("\nFIRST IMAGES\n")


    for url in urls[:10]:

        print(url)


# ==================================================
# summary
# ==================================================

good_rows = len(df) - max(

    len(missing_urls),

    len(zero_images),

    len(invalid_rows),

    len(duplicate_articles)

)


print("\nlikely good rows\n")

print(good_rows)


print("\nfinished")

In [ ]:
import pandas as pd
import ast


# ==================================================
# show full urls
# ==================================================

pd.set_option(

    "display.max_colwidth",

    None

)


# ==================================================
# load dataset
# ==================================================

df = pd.read_csv(
    "spiegel_clean_article_images.csv"
)


# ==================================================
# flatten images
# ==================================================

all_rows = []


for idx, row in df.iterrows():

    article_url = row["article_url"]


    try:

        urls = ast.literal_eval(
            row["image_urls"]
        )

    except:

        continue


    for url in urls:

        all_rows.append({

            "article_url": article_url,

            "image_url": url

        })


images_df = pd.DataFrame(
    all_rows
)


# ==================================================
# frequencies
# ==================================================

freq = (

    images_df["image_url"]
    .value_counts()

)


freq_df = freq.reset_index()

freq_df.columns = [

    "image_url",
    "count"

]


# ==================================================
# helper
# ==================================================

def inspect_range(

    min_count,
    max_count,
    sample_size=20

):

    subset = freq_df[

        (freq_df["count"] >= min_count)

        &

        (freq_df["count"] < max_count)

    ]


    print("\n====================================\n")

    print(f"RANGE {min_count}-{max_count}")

    print("\nimages in range\n")

    print(len(subset))


    if len(subset) == 0:

        return


    sample_n = min(

        sample_size,

        len(subset)

    )


    sample = subset.sample(
        sample_n
    )


    print("\nSAMPLE\n")


    for idx, row in sample.iterrows():

        print("\nCOUNT\n")

        print(row["count"])


        print("\nIMAGE\n")

        print(row["image_url"])


        print("\n----------------------------\n")


# ==================================================
# inspect ranges
# ==================================================

inspect_range(5, 10)

inspect_range(10, 20)

inspect_range(30, 40)

In [ ]:
import pandas as pd
import ast


# ==================================================
# settings
# ==================================================

input_file = "spiegel_clean_article_images.csv"

output_file = "spiegel_clean_article_images_filtered.csv"

repeat_threshold = 35


# ==================================================
# load dataset
# ==================================================

df = pd.read_csv(input_file)


print("\nrows\n")

print(len(df))


# ==================================================
# flatten all images
# ==================================================

all_images = []


for idx, row in df.iterrows():

    try:

        urls = ast.literal_eval(
            row["image_urls"]
        )

    except:

        continue


    for url in urls:

        all_images.append(url)


# ==================================================
# image frequencies
# ==================================================

image_counts = pd.Series(

    all_images

).value_counts()


print("\nmost repeated images\n")

print(

    image_counts.head(20)

)


# ==================================================
# images to remove
# ==================================================

bad_images = set(

    image_counts[
        image_counts > repeat_threshold
    ].index

)


print("\nimages removed\n")

print(len(bad_images))


# ==================================================
# filter dataset
# ==================================================

new_rows = []


removed_total = 0


for idx, row in df.iterrows():

    try:

        urls = ast.literal_eval(
            row["image_urls"]
        )

    except:

        urls = []


    # keep only good images

    filtered = [

        url for url in urls

        if url not in bad_images

    ]


    removed_total += (

        len(urls) - len(filtered)

    )


    row["image_urls"] = str(
        filtered
    )

    row["image_count"] = len(
        filtered
    )

    row["has_images"] = (

        len(filtered) > 0

    )


    new_rows.append(row)


# ==================================================
# final dataframe
# ==================================================

final_df = pd.DataFrame(
    new_rows
)


# ==================================================
# diagnostics
# ==================================================

print("\nremoved image instances\n")

print(removed_total)


print("\narticles with images\n")

print(

    final_df["has_images"]
    .value_counts()

)


print("\nimage count stats\n")

print(

    final_df["image_count"]
    .describe()

)


print("\nrows without images\n")

print(

    (
        final_df["image_count"] == 0
    ).sum()

)


# ==================================================
# save
# ==================================================

final_df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nfiltered dataset saved\n")

print(output_file)


print("\nfinished")

In [ ]:
import pandas as pd
import ast


# load dataset

df = pd.read_csv(
    "spiegel_clean_article_images_filtered.csv"
)


# collect all images

all_images = []


for idx, row in df.iterrows():

    try:

        urls = ast.literal_eval(
            row["image_urls"]
        )

    except:

        continue


    all_images.extend(urls)


# unique images

unique_images = set(
    all_images
)


print("\ntotal image references\n")

print(len(all_images))


print("\nunique images\n")

print(len(unique_images))

## Image download by 5k

In [ ]:
# import pandas as pd
import ast
import requests
import os
import hashlib
import pandas as pd
from tqdm import tqdm


# ==================================================
# settings
# ==================================================

input_file = "spiegel_clean_article_images_filtered.csv"

output_dir = "spiegel_unique_images"

failed_file = "spiegel_failed_unique_downloads_batch1.csv"


batch_size = 60000

batch_start = 15000

timeout = 30


# ==================================================
# create folder
# ==================================================

os.makedirs(

    output_dir,

    exist_ok=True

)


# ==================================================
# load dataset
# ==================================================

df = pd.read_csv(input_file)


print("\nrows\n")

print(len(df))


# ==================================================
# collect unique images
# ==================================================

all_images = []


for idx, row in df.iterrows():

    try:

        urls = ast.literal_eval(
            row["image_urls"]
        )

    except:

        continue


    all_images.extend(urls)


# unique only

unique_images = sorted(

    list(set(all_images))

)


print("\nall unique images\n")

print(len(unique_images))


# ==================================================
# batch
# ==================================================

batch_end = batch_start + batch_size


batch_images = unique_images[
    batch_start:batch_end
]


print("\nbatch images\n")

print(len(batch_images))


print("\nbatch range\n")

print(batch_start)

print(batch_end)


# ==================================================
# headers
# ==================================================

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )

}


# ==================================================
# helper
# ==================================================

def make_filename(url):

    url_hash = hashlib.md5(

        url.encode()

    ).hexdigest()


    lower = url.lower()


    if ".jpg" in lower:

        ext = ".jpg"

    elif ".jpeg" in lower:

        ext = ".jpeg"

    elif ".png" in lower:

        ext = ".png"

    elif ".webp" in lower:

        ext = ".webp"

    elif ".avif" in lower:

        ext = ".avif"

    else:

        ext = ".jpg"


    return url_hash + ext


# ==================================================
# download
# ==================================================

failed = []

downloaded = 0


for url in tqdm(batch_images):

    try:

        filename = make_filename(
            url
        )


        filepath = os.path.join(

            output_dir,

            filename

        )


        # skip existing

        if os.path.exists(filepath):

            downloaded += 1

            continue


        response = requests.get(

            url,

            headers=headers,

            timeout=timeout,

            stream=True

        )


        if response.status_code != 200:

            failed.append({

                "image_url": url,

                "status_code": response.status_code

            })

            continue


        with open(

            filepath,

            "wb"

        ) as f:

            for chunk in response.iter_content(
                8192
            ):

                f.write(chunk)


        downloaded += 1


    except Exception as e:

        failed.append({

            "image_url": url,

            "error": str(e)

        })


# ==================================================
# failed log
# ==================================================

failed_df = pd.DataFrame(
    failed
)


failed_df.to_csv(

    failed_file,

    index=False,

    encoding="utf-8-sig"

)


# ==================================================
# diagnostics
# ==================================================

print("\ndownloaded\n")

print(downloaded)


print("\nfailed\n")

print(len(failed))


print("\noutput dir\n")

print(output_dir)


print("\nfailed log\n")

print(failed_file)


print("\nfinished")

# Download missing dates for Spiegel

In [ ]:
# Fetch datePublished from Spiegel article pages (JSON-LD) for every article_url already in
# spiegel_final_data.csv, then merge the recovered dates back into that file and save.
# Only downloads dates — no text/images are re-fetched.

import json
import requests
import pandas as pd

from bs4 import BeautifulSoup
from tqdm import tqdm

from concurrent.futures import ThreadPoolExecutor, as_completed

# =====================================================
# SETTINGS
# =====================================================

INPUT_FILE = "spiegel_final_data.csv"
OUTPUT_FILE = "spiegel_final_data_with_dates.csv"
FAILED_FILE = "spiegel_dates_failed.csv"

MAX_WORKERS = 10
TIMEOUT = 30

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )
}

# =====================================================
# JSON-LD DATE EXTRACTION
# =====================================================

def extract_date_published(html):
    soup = BeautifulSoup(html, "lxml")
    jsonld_blocks = soup.find_all("script", type="application/ld+json")

    for block in jsonld_blocks:
        try:
            data = json.loads(block.string)
        except Exception:
            continue

        candidates = data if isinstance(data, list) else [data]

        for item in candidates:
            if isinstance(item, dict) and "datePublished" in item:
                return item["datePublished"]

    return None


def fetch_date(article_url):
    try:
        response = requests.get(article_url, headers=HEADERS, timeout=TIMEOUT)
        response.raise_for_status()
        date = extract_date_published(response.text)
        return {"article_url": article_url, "date": date}
    except Exception as exc:
        return {"article_url": article_url, "date": None, "error": str(exc)}


# =====================================================
# LOAD CORPUS
# =====================================================

df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"\narticles to fetch dates for: {len(df)}\n")

results = []
failed = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(fetch_date, url): url
        for url in df["article_url"].dropna().unique()
    }

    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        if result.get("date"):
            results.append(result)
        else:
            failed.append(result)

print(f"\nrecovered dates: {len(results)}")
print(f"failed: {len(failed)}")

# =====================================================
# MERGE BACK AND SAVE
# =====================================================

dates_df = pd.DataFrame(results)

df = df.drop(columns=["date"]).merge(dates_df, on="article_url", how="left")
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

if failed:
    pd.DataFrame(failed).to_csv(FAILED_FILE, index=False, encoding="utf-8-sig")
    print(f"failed rows saved to {FAILED_FILE}")

In [ ]:
# Retry failed Spiegel date fetches, slower and single-threaded to avoid the connection
# throttling. Results are appended into the SAME two files every run (not new files each time):
# spiegel_dates_recovered.csv accumulates every date ever recovered, spiegel_dates_failed.csv
# always holds only what's still failing. Safe to interrupt and re-run — it resumes automatically
# by skipping URLs already in spiegel_dates_recovered.csv, and saves progress periodically.

import os
import time
import random
import json
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

RECOVERED_FILE = "spiegel_dates_recovered.csv"
FAILED_FILE = "spiegel_dates_failed.csv"
SAVE_EVERY = 200

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
}

def extract_date_published(html):
    soup = BeautifulSoup(html, "lxml")
    for block in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(block.string)
        except Exception:
            continue
        items = data if isinstance(data, list) else [data]
        for item in items:
            if isinstance(item, dict) and "datePublished" in item:
                return item["datePublished"]
    return None

# ---------- load existing progress ----------
if os.path.exists(RECOVERED_FILE):
    recovered_df = pd.read_csv(RECOVERED_FILE)
    already_done = set(recovered_df["article_url"])
    print(f"already recovered: {len(already_done)}")
else:
    recovered_df = pd.DataFrame(columns=["article_url", "date"])
    already_done = set()

failed_df = pd.read_csv(FAILED_FILE)
urls_to_retry = [u for u in failed_df["article_url"] if u not in already_done]
print(f"retrying: {len(urls_to_retry)}")

# ---------- retry loop with periodic save ----------
session = requests.Session()
session.headers.update(HEADERS)

new_results = []
still_failed = []

def flush_progress():
    global recovered_df
    if new_results:
        recovered_df = pd.concat([recovered_df, pd.DataFrame(new_results)], ignore_index=True)
        recovered_df = recovered_df.drop_duplicates(subset=["article_url"])
        recovered_df.to_csv(RECOVERED_FILE, index=False, encoding="utf-8-sig")
    pd.DataFrame(still_failed).to_csv(FAILED_FILE, index=False, encoding="utf-8-sig")

for i, url in enumerate(tqdm(urls_to_retry)):
    try:
        r = session.get(url, timeout=30)
        if r.status_code != 200:
            still_failed.append({"article_url": url, "error": f"status_{r.status_code}"})
        else:
            date = extract_date_published(r.text)
            if date:
                new_results.append({"article_url": url, "date": date})
            else:
                still_failed.append({"article_url": url, "error": "no_date_found"})
    except Exception as e:
        still_failed.append({"article_url": url, "error": str(e)})

    time.sleep(random.uniform(1.0, 2.5))

    if (i + 1) % SAVE_EVERY == 0:
        flush_progress()

flush_progress()

print(f"\nnewly recovered this run: {len(new_results)}")
print(f"total recovered so far: {len(recovered_df)}")
print(f"still failed: {len(still_failed)}")

# после надо привести даты к одному формату со всеми 

In [ ]:
import re
import json
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# =====================================================
# SETTINGS
# =====================================================

INPUT_FILE = "spiegel_final_data.csv"
OUTPUT_FILE = "spiegel_final_data.csv"          # перезаписываем на месте
BACKUP_FILE = "spiegel_final_data_backup_before_datefix.csv"
FAILED_FILE = "spiegel_dates_failed.csv"

MAX_WORKERS = 10
TIMEOUT = 30
TARGET_FORMAT = "%Y-%m-%d %H:%M:%S"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
    )
}

# =====================================================
# FETCH DATE FOR ONE ARTICLE
# =====================================================

def fetch_date(article_url):
    try:
        r = requests.get(article_url, headers=HEADERS, timeout=TIMEOUT)
        if r.status_code != 200:
            return (False, {"article_url": article_url, "error": f"status_{r.status_code}"})

        soup = BeautifulSoup(r.text, "lxml")

        # 1) JSON-LD datePublished (NewsArticle или любой другой @type)
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.get_text(strip=True))
            except Exception:
                continue
            items = data if isinstance(data, list) else [data]
            for item in items:
                if isinstance(item, dict) and item.get("datePublished"):
                    return (True, {"article_url": article_url, "date": item["datePublished"]})

        # 2) og / article meta tags
        for prop in ["article:published_time", "og:updated_time"]:
            m = soup.find("meta", property=prop)
            if m and m.get("content"):
                return (True, {"article_url": article_url, "date": m["content"]})

        # 3) content:publishDate / другие name-мета-теги
        for meta_name in ["content:publishDate", "date", "sailthru.date"]:
            m = soup.find("meta", attrs={"name": meta_name})
            if m and m.get("content"):
                return (True, {"article_url": article_url, "date": m["content"]})

        # 4) <time datetime="..."> тег
        t = soup.find("time")
        if t and t.get("datetime"):
            return (True, {"article_url": article_url, "date": t["datetime"]})

        return (False, {"article_url": article_url, "error": "no_date_found"})
    except Exception as e:
        return (False, {"article_url": article_url, "error": str(e)})


# =====================================================
# RUN
# =====================================================

df = pd.read_csv(INPUT_FILE)
print(f"loaded {len(df)} rows")

# бэкап до любых изменений
Path(BACKUP_FILE).write_bytes(Path(INPUT_FILE).read_bytes())
print(f"backup written: {BACKUP_FILE}")

urls_to_fetch = df["article_url"].tolist()
print(f"fetching dates for: {len(urls_to_fetch)}")

rows = []
failed = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_date, url) for url in urls_to_fetch]
    for future in tqdm(as_completed(futures), total=len(futures)):
        success, result = future.result()
        (rows if success else failed).append(result)

print(f"\nrecovered: {len(rows)}")
print(f"still failed: {len(failed)}")

dates_df = pd.DataFrame(rows)

# мерджим восстановленные даты обратно
df = df.drop(columns=["date"], errors="ignore").merge(dates_df, on="article_url", how="left")

# сразу приводим к единому формату, как у остальных 7 источников
dt = pd.to_datetime(df["date"], errors="coerce", utc=True, format="mixed")
print(f"\nunparsed after standardizing format: {dt.isna().sum()}")
df["date"] = dt.dt.strftime(TARGET_FORMAT)

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"saved: {OUTPUT_FILE}")

if failed:
    pd.DataFrame(failed).to_csv(FAILED_FILE, index=False, encoding="utf-8-sig")
    print(f"failed rows saved to {FAILED_FILE}")